# ✈️ Visual Co-Pilot — Aviation VQA System
### Capstone Project | Radar Dataset + Hybrid AI Engine

**Run cells top to bottom. GPU runtime strongly recommended.**

> **Before starting:** Runtime → Change runtime type → **T4 GPU**


## ⚙️ Step 1 — Install All Packages
> Run this first every session. Wait for it to fully complete.

In [ ]:
# CELL 1 — Install everything
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip"] + list(args),
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Uninstalling stale chromadb...")
pip("uninstall", "-y", "chromadb")

print("Installing packages...")
pip("install", "-q",
    "openai-clip", "torch", "torchvision", "transformers",
    "chromadb", "requests", "pillow", "matplotlib",
    "numpy", "pandas", "scikit-learn",
    "sounddevice", "SpeechRecognition", "scipy",
    "tqdm", "ipywidgets", "faster-whisper")

import torch, chromadb
print(f"chromadb : {chromadb.__version__} ✓")
print(f"PyTorch  : {torch.__version__} ✓")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
print("Done ✓")

Uninstalling stale chromadb...
Installing packages...
chromadb : 1.5.9 ✓
PyTorch  : 2.11.0+cu128 ✓
CUDA     : True
GPU      : Tesla T4
Done ✓


## 📦 Step 2 — Upload Zip & Mount Drive

In [ ]:
# ============================================================
# CELL 2 — Mount Drive + Extract ZIP + Fix all imports
# ============================================================

import os
import sys
import zipfile
import json
import shutil

from google.colab import drive, files


# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------

drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/aviation_vqa_output"
PROC_DIR = os.path.join(BASE_DIR, "data", "processed")
RAW_DIR  = os.path.join(BASE_DIR, "data", "raw")

# Create required Drive directories
for d in [
    "data/raw/images",
    "data/processed/images",
    "data/annotations",
    "embeddings",
    "chromadb_store",
    "models",
    "evaluation_results",
    "demo_sessions",
]:
    os.makedirs(os.path.join(BASE_DIR, d), exist_ok=True)

print(f"Drive mounted ✓")
print(f"BASE_DIR = {BASE_DIR}")


# ------------------------------------------------------------
# 2. Upload ZIP
# ------------------------------------------------------------

print("\nUpload aviation_vqa.zip now ↓")

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No ZIP file was uploaded.")

zip_name = next(iter(uploaded.keys()))
zip_path = os.path.abspath(zip_name)

print(f"ZIP received ✓  {zip_path}")


# ------------------------------------------------------------
# 3. Clean previous temporary extraction
# ------------------------------------------------------------

EXTRACT_DIR = "/content/aviation_vqa"

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)

os.makedirs(EXTRACT_DIR, exist_ok=True)

print(f"Temporary extraction directory ready ✓")
print(f"EXTRACT_DIR = {EXTRACT_DIR}")


# ------------------------------------------------------------
# 4. Extract ZIP
# ------------------------------------------------------------

print("\nExtracting ZIP...")

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(EXTRACT_DIR)

print("Extracted ✓")


# ------------------------------------------------------------
# 5. Detect actual project root
# ------------------------------------------------------------

print("\nFinding project files...")

# Look for a directory containing one or more of the
# expected Aviation VQA modules.
EXPECTED_MODULES = {
    "radar_generation",
    "data_collection",
    "qa_generation",
    "preprocessing",
    "embeddings",
    "chromadb_store",
    "models",
    "inference",
    "evaluation",
    "voice",
}

PROJECT_ROOT = None

for root, dirs, files_list in os.walk(EXTRACT_DIR):

    dir_names = set(dirs)

    # Strong match: project contains several expected modules
    matches = EXPECTED_MODULES.intersection(dir_names)

    if len(matches) >= 3:
        PROJECT_ROOT = root
        break


# Fallback: if the ZIP itself contains the modules directly
if PROJECT_ROOT is None:
    root_dirs = set(os.listdir(EXTRACT_DIR))

    matches = EXPECTED_MODULES.intersection(root_dirs)

    if len(matches) >= 3:
        PROJECT_ROOT = EXTRACT_DIR


if PROJECT_ROOT is None:

    print("\nCould not automatically identify project root.")

    print("\nExtracted directory structure:")
    for root, dirs, files_list in os.walk(EXTRACT_DIR):
        level = root.replace(EXTRACT_DIR, "").count(os.sep)
        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/")

        for f in files_list[:10]:
            print(f"{indent}  {f}")

    raise RuntimeError(
        "\nCould not find Aviation VQA project root. "
        "Check the ZIP structure printed above."
    )


print(f"\nProject root found ✓")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


# ------------------------------------------------------------
# 6. Verify important modules
# ------------------------------------------------------------

print("\nChecking project structure...")

for module in sorted(EXPECTED_MODULES):
    module_path = os.path.join(PROJECT_ROOT, module)

    if os.path.isdir(module_path):
        print(f"  {module}/ ✓")
    else:
        print(f"  {module}/ — not present")


# ------------------------------------------------------------
# 7. Write __init__.py into Python modules
# ------------------------------------------------------------

MODULES = [
    "radar_generation",
    "data_collection",
    "qa_generation",
    "preprocessing",
    "embeddings",
    "chromadb_store",
    "models",
    "inference",
    "evaluation",
    "voice",
    "gui",
    "utils",
]

for mod in MODULES:

    module_dir = os.path.join(PROJECT_ROOT, mod)

    # Only create __init__.py if the module directory exists
    if os.path.isdir(module_dir):

        init_path = os.path.join(module_dir, "__init__.py")

        if not os.path.exists(init_path):
            with open(init_path, "w", encoding="utf-8") as f:
                f.write(f"# {mod}\n")

print("\n__init__.py files checked/written ✓")


# ------------------------------------------------------------
# 8. Add project root to sys.path
# ------------------------------------------------------------

# Remove previous Aviation VQA paths
sys.path = [
    p for p in sys.path
    if not os.path.abspath(p).startswith(
        os.path.abspath(EXTRACT_DIR)
    )
]

# Add the ACTUAL project root
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("\nsys.path configured ✓")
print(f"Python project path = {PROJECT_ROOT}")


# ------------------------------------------------------------
# 9. Test imports
# ------------------------------------------------------------

print("\nTesting imports...")

errors = []

tests = [
    ("radar_generation.radar_renderer", "RadarRenderer"),
    ("data_collection.opensky_collector", "SyntheticOpenSkyCollector"),
    ("qa_generation.qa_generator", "RadarQAGenerator"),
    ("preprocessing.dataset_builder", "DatasetBuilder"),
    ("embeddings.clip_embedder", "CLIPEmbedder"),
    ("chromadb_store.chroma_manager", "RadarChromaManager"),
    ("models.vqa_model", "RadarVQAModel"),
    ("models.trainer", "VQATrainer"),
    ("models.dataset", "RadarVQADataset"),
    ("inference.hybrid_engine", "HybridInferenceEngine"),
    ("evaluation.evaluator", "VQAEvaluator"),
    ("voice.voice_query", "TextQueryInterface"),
]

for module_name, class_name in tests:

    try:

        module = __import__(
            module_name,
            fromlist=[class_name]
        )

        getattr(module, class_name)

        print(f"  {module_name} ✓")

    except Exception as e:

        print(f"  {module_name} ✗ — {e}")

        errors.append(
            (module_name, str(e))
        )


# ------------------------------------------------------------
# 10. Load raw_data from Drive if available
# ------------------------------------------------------------

raw_manifest = os.path.join(
    RAW_DIR,
    "raw_frames.json"
)

if os.path.exists(raw_manifest):

    with open(
        raw_manifest,
        "r",
        encoding="utf-8"
    ) as f:

        raw_data = json.load(f)

    print(
        f"\nraw_data loaded: "
        f"{len(raw_data)} frames ✓"
    )

else:

    raw_data = []

    print(
        "\nraw_data not on Drive yet "
        "— Step 3 will generate it"
    )


# ------------------------------------------------------------
# 11. Final status
# ------------------------------------------------------------

print("\n" + "=" * 60)

if errors:

    print(
        f"⚠️ {len(errors)} import(s) failed."
    )

    print("\nFailed imports:")

    for module_name, error in errors:
        print(f"  • {module_name}")
        print(f"    {error}")

else:

    print("🎉 ALL IMPORTS OK!")
    print("You can proceed to Step 3.")

print("=" * 60)

Mounted at /content/drive
Drive mounted ✓
BASE_DIR = /content/drive/MyDrive/aviation_vqa_output

Upload aviation_vqa.zip now ↓


Saving aviation_vqa.zip to aviation_vqa.zip
ZIP received ✓  /content/aviation_vqa.zip
Temporary extraction directory ready ✓
EXTRACT_DIR = /content/aviation_vqa

Extracting ZIP...
Extracted ✓

Finding project files...

Project root found ✓
PROJECT_ROOT = /content/aviation_vqa

Checking project structure...
  chromadb_store/ ✓
  data_collection/ ✓
  embeddings/ ✓
  evaluation/ ✓
  inference/ ✓
  models/ ✓
  preprocessing/ ✓
  qa_generation/ ✓
  radar_generation/ ✓
  voice/ ✓

__init__.py files checked/written ✓

sys.path configured ✓
Python project path = /content/aviation_vqa

Testing imports...
  radar_generation.radar_renderer ✓
  data_collection.opensky_collector ✓
  qa_generation.qa_generator ✓
  preprocessing.dataset_builder ✓
  embeddings.clip_embedder ✓
  chromadb_store.chroma_manager ✓
  models.vqa_model ✓
  models.trainer ✓
  models.dataset ✓
  inference.hybrid_engine ✓
  evaluation.evaluator ✓
  voice.voice_query ✓

raw_data loaded: 2000 frames ✓

🎉 ALL IMPORTS OK!
You can pr

## 📡 Step 3 — Collect 2000 Unique Aircraft Frames
> Uses OpenSky API if reachable, else synthetic data automatically.

In [ ]:
import sys, os, json, hashlib
sys.path.insert(0, '/content/aviation_vqa')

RAW_DIR  = os.path.join(BASE_DIR, 'data/raw')
MANIFEST = os.path.join(RAW_DIR, 'raw_frames.json')
os.makedirs(RAW_DIR, exist_ok=True)

if os.path.exists(MANIFEST):
    with open(MANIFEST) as f: raw_data = json.load(f)
    print(f'Loaded {len(raw_data)} existing frames  ✓')
else:
    USE_SYNTHETIC = False
    if not USE_SYNTHETIC:
        try:
            import requests
            r = requests.get('https://opensky-network.org/api/states/all',
                             params={'lamin':36,'lomin':-10,'lamax':60,'lomax':25}, timeout=10)
            r.raise_for_status()
            from data_collection.opensky_collector import OpenSkyCollector
            collector = OpenSkyCollector(output_dir=RAW_DIR, num_frames=2000,
                                        aircraft_per_frame=(4,10), sleep_between=2.0)
            print('Using live OpenSky API ...')
        except Exception as e:
            print(f'OpenSky unavailable ({e}). Using synthetic.')
            USE_SYNTHETIC = True
    if USE_SYNTHETIC:
        from data_collection.opensky_collector import SyntheticOpenSkyCollector
        collector = SyntheticOpenSkyCollector(output_dir=RAW_DIR, num_frames=2000,
                                              aircraft_per_frame=(4,10), sleep_between=0.0)
    raw_data = collector.collect()
    print(f'Collected {len(raw_data)} unique frames  ✓')


Loaded 2000 existing frames  ✓


## 🖼️ Step 4 — Render 2000 Radar Images (224×224 PNG)

In [ ]:
import os, json, sys
sys.path.insert(0, '/content/aviation_vqa')
from radar_generation.radar_renderer import RadarRenderer
from PIL import Image
import matplotlib.pyplot as plt, random

IMG_DIR = os.path.join(BASE_DIR, 'data/raw/images')
REN_MAN = os.path.join(BASE_DIR, 'data/raw/rendered_frames.json')

if os.path.exists(REN_MAN):
    with open(REN_MAN) as f: rendered = json.load(f)
    print(f'Loaded {len(rendered)} existing rendered frames  ✓')
else:
    renderer = RadarRenderer(output_dir=IMG_DIR, image_size=224)
    rendered = renderer.render_all(raw_data)
    with open(REN_MAN, 'w') as f: json.dump(rendered, f, indent=2)
    print(f'Rendered {len(rendered)} radar images  ✓')

samples = random.sample(rendered, min(4, len(rendered)))
fig, axes = plt.subplots(1, len(samples), figsize=(4*len(samples), 4))
if len(samples)==1: axes=[axes]
for ax, fr in zip(axes, samples):
    ax.imshow(Image.open(fr['image_path']))
    ax.set_title(f"{fr['frame_id']}\n{len(fr['aircraft'])} aircraft", fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Radar Images'); plt.tight_layout(); plt.show()
print(f'Total images: {len(rendered)}')


Loaded 2000 existing rendered frames  ✓
Total images: 2000


## ❓ Step 5 — Generate QA Pairs (7 per image ≈ 14,000 total)

In [ ]:
import os, json, sys
sys.path.insert(0, '/content/aviation_vqa')
from qa_generation.qa_generator import RadarQAGenerator
from collections import Counter

ANNO_PATH = os.path.join(BASE_DIR, 'data/annotations/annotations.jsonl')

if os.path.exists(ANNO_PATH):
    with open(ANNO_PATH) as f:
        annotations = [json.loads(l) for l in f if l.strip()]
    print(f'Loaded {len(annotations)} existing QA pairs  ✓')
else:
    for fr in rendered:
        fr['image_path'] = os.path.join(BASE_DIR, 'data/raw/images', f"{fr['frame_id']}.png")
    qa_gen = RadarQAGenerator()
    annotations = qa_gen.generate_all(frames=rendered, qa_per_image=7, output_path=ANNO_PATH)
    print(f'Generated {len(annotations)} QA pairs  ✓')

dist = Counter(r['question_type'] for r in annotations)
print(f'Total QA pairs: {len(annotations):,}')
print('\nQuestion type distribution:')
for t, c in sorted(dist.items(), key=lambda x: -x[1]):
    print(f'  {t:<30} {c:>6}')


Loaded 14000 existing QA pairs  ✓
Total QA pairs: 14,000

Question type distribution:
  region_density                   1033
  positional_top                   1026
  comparative_highest              1023
  altitude_lowest                  1021
  heading                          1013
  threshold                        1013
  positional_left                  1009
  positional_center                1007
  comparative_lowest               1002
  boolean_count                     985
  positional_bottom                 983
  altitude_value                    971
  counting                          964
  positional_right                  950


## 🔧 Step 6 — Preprocess & 70/30 Train/Test Split

In [ ]:
import os, json, sys
sys.path.insert(0, '/content/aviation_vqa')
from preprocessing.dataset_builder import DatasetBuilder

PROC_DIR = os.path.join(BASE_DIR, 'data/processed')
TRAIN_J  = os.path.join(PROC_DIR, 'train.json')

if os.path.exists(TRAIN_J):
    with open(os.path.join(PROC_DIR,'answer_index.json')) as f: answer_index = json.load(f)
    with open(TRAIN_J) as f: tr = json.load(f)
    with open(os.path.join(PROC_DIR,'test.json')) as f: te = json.load(f)
    print(f'Loaded: train={len(tr)}, test={len(te)}, classes={len(answer_index)}  ✓')
else:
    builder = DatasetBuilder(
        raw_image_dir=os.path.join(BASE_DIR,'data/raw/images'),
        annotation_path=ANNO_PATH, processed_dir=PROC_DIR, train_ratio=0.7)
    stats = builder.build()
    print(f'Train: {stats["train"]:,} | Test: {stats["test"]:,} | Classes: {stats["num_classes"]}')
    with open(os.path.join(PROC_DIR,'answer_index.json')) as f: answer_index = json.load(f)
print(f'Answer vocab: {len(answer_index)} classes  ✓')


Loaded: train=9800, test=4200, classes=2119  ✓
Answer vocab: 2119 classes  ✓


## 🚫 CLIP / Hybrid pipeline discarded
> The former Steps 7–10 (CLIP embeddings, ChromaDB vector store, CLIP-based VQA training, and the hybrid CLIP+ChromaDB inference engine) have been removed. This build answers questions using **only the ResNet50 CNN pipeline** below (`cnn_engine`) — no CLIP, no embeddings, no vector database anywhere in the voice, text, or detection interfaces.

In [ ]:
import subprocess, sys
# torchvision already installed but make sure
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torchvision"])

import os, json, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from typing import Dict, List, Tuple
import numpy as np

# Re-defining BASE_DIR to ensure it's available if cells are run out of order or after a kernel restart
BASE_DIR = '/content/drive/MyDrive/aviation_vqa_output'

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
PROC_DIR = os.path.join(BASE_DIR, "data/processed")

print(f"Device  : {DEVICE}")
print(f"PyTorch : {torch.__version__}")
print("CNN setup ready ✓")

Device  : cuda
PyTorch : 2.11.0+cu128
CNN setup ready ✓


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CNN DIRECT IMAGE READING — Cell B: Model Classes
# ═══════════════════════════════════════════════════════════════════

# ── Dataset: reads raw pixels, no CLIP ───────────────────────────
class CNNVQADataset(Dataset):
    """
    Reads images as raw RGB pixel arrays.
    Questions tokenised as character indices — no external tokeniser.
    """
    CHARS    = list("abcdefghijklmnopqrstuvwxyz0123456789 ?,'.")
    CHAR2IDX = {c: i+1 for i, c in enumerate(CHARS)}
    MAX_Q    = 60

    def __init__(self, json_path, split="train", image_size=224):
        with open(json_path) as f:
            self.records = json.load(f)
        norm = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std= [0.229, 0.224, 0.225])
        if split == "train":
            self.tf = transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.RandomHorizontalFlip(0.3),
                transforms.ColorJitter(brightness=0.15, contrast=0.15),
                transforms.ToTensor(), norm])
        else:
            self.tf = transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(), norm])

    def __len__(self): return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        # Read raw image pixels
        try:    img = Image.open(rec["image_path"]).convert("RGB")
        except: img = Image.new("RGB", (224,224), (0,26,0))
        img_t = self.tf(img)   # (3, 224, 224) pixel tensor

        # Tokenise question as character indices
        q = rec["question"].lower()[:self.MAX_Q]
        q_tok = [self.CHAR2IDX.get(c, 0) for c in q]
        q_tok += [0] * (self.MAX_Q - len(q_tok))
        q_t = torch.tensor(q_tok, dtype=torch.long)

        label = torch.tensor(rec["label_id"], dtype=torch.long)
        return img_t, q_t, label


# ── CNN Visual Encoder: ResNet50 reads pixel patterns ─────────────
class CNNVisualEncoder(nn.Module):
    """
    ResNet50 backbone — reads raw pixels directly.
    Early layers detect edges and blip shapes.
    Mid layers detect spatial clusters (left/right/top/bottom).
    Late layers detect high-level features (blip count, density).
    """
    def __init__(self, out_dim=512):
        super().__init__()
        backbone      = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(backbone.children())[:-1])  # → (B,2048,1,1)
        self.proj     = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, out_dim),
            nn.LayerNorm(out_dim),
            nn.GELU(),
            nn.Dropout(0.15))

    def forward(self, x):
        return self.proj(self.features(x))   # (B, out_dim)


# ── Question Encoder: char-level GRU, no CLIP ─────────────────────
class QuestionEncoder(nn.Module):
    def __init__(self, vocab=40, embed=64, hidden=256, out_dim=512):
        super().__init__()
        self.embed = nn.Embedding(vocab+1, embed, padding_idx=0)
        self.gru   = nn.GRU(embed, hidden, num_layers=2,
                            batch_first=True, dropout=0.2, bidirectional=True)
        self.proj  = nn.Sequential(
            nn.Linear(hidden*2, out_dim),
            nn.LayerNorm(out_dim), nn.GELU(), nn.Dropout(0.15))

    def forward(self, x):
        emb = self.embed(x)
        _, h = self.gru(emb)
        return self.proj(torch.cat([h[-2], h[-1]], dim=-1))


# ── Full CNN VQA Model ─────────────────────────────────────────────
class CNNVQAModel(nn.Module):
    """
    Direct pixel-reading VQA.
    Pipeline: raw pixels → ResNet50 CNN → cross-attention
              with question → answer classifier.
    No CLIP, no embeddings anywhere.
    """
    def __init__(self, num_classes, device="cpu", feat_dim=512):
        super().__init__()
        self.device       = device
        self.visual_enc   = CNNVisualEncoder(out_dim=feat_dim)
        self.question_enc = QuestionEncoder(out_dim=feat_dim)
        self.cross_attn   = nn.MultiheadAttention(
            feat_dim, num_heads=8, dropout=0.1, batch_first=True)
        self.norm1      = nn.LayerNorm(feat_dim)
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim*2, feat_dim), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(feat_dim, feat_dim//2), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(feat_dim//2, num_classes))
        self.to(device)

    def forward(self, images, questions):
        images    = images.to(self.device)
        questions = questions.to(self.device)
        v = self.visual_enc(images)        # pixels → features
        q = self.question_enc(questions)   # chars  → features
        # Cross-attention: visual attends to question
        v_s = v.unsqueeze(1)
        q_s = q.unsqueeze(1)
        att, _ = self.cross_attn(v_s, q_s, q_s)
        fused   = self.norm1(v_s + att).squeeze(1)
        return self.classifier(torch.cat([fused, q], dim=-1))

    def predict(self, images, questions, answer_index):
        self.eval()
        with torch.no_grad():
            logits = self.forward(images, questions)
            probs  = F.softmax(logits, dim=-1)
            ids    = logits.argmax(-1).cpu().tolist()
            confs  = probs.max(-1).values.cpu().tolist()
        idx2ans = {v: k for k, v in answer_index.items()}
        return [idx2ans.get(i,"unknown") for i in ids], confs

    def save(self, path): torch.save(self.state_dict(), path)
    def load(self, path):
        self.load_state_dict(
            torch.load(path, map_location=self.device, weights_only=True))
        return self


# ── Trainer ───────────────────────────────────────────────────────
class CNNVQATrainer:
    def __init__(self, model, train_loader, val_loader,
                 device="cpu", lr=1e-4, epochs=15, save_dir="models"):
        self.model        = model
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.device       = device
        self.epochs       = epochs
        self.save_dir     = save_dir
        os.makedirs(save_dir, exist_ok=True)
        trainable      = [p for p in model.parameters() if p.requires_grad]
        self.optim     = torch.optim.AdamW(trainable, lr=lr, weight_decay=1e-2)
        def lr_fn(ep):
            if ep < 2: return (ep+1)/2
            t = (ep-2)/max(1, epochs-2)
            return 0.5*(1+torch.cos(torch.tensor(t*3.14159)).item())
        self.sched     = torch.optim.lr_scheduler.LambdaLR(self.optim, lr_fn)
        self.criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    def train(self):
        history  = {"train_loss":[],"train_acc":[],"val_loss":[],"val_acc":[]}
        best_acc = 0.0
        best_path = os.path.join(self.save_dir, "cnn_best_model.pth")
        for ep in range(1, self.epochs+1):
            tl, ta = self._epoch(self.train_loader, train=True)
            vl, va = self._epoch(self.val_loader,   train=False)
            self.sched.step()
            history["train_loss"].append(tl); history["train_acc"].append(ta)
            history["val_loss"].append(vl);   history["val_acc"].append(va)
            print(f"Ep {ep:02d}/{self.epochs} | "
                  f"train loss={tl:.4f} acc={ta:.2f}% | "
                  f"val loss={vl:.4f} acc={va:.2f}%")
            if va > best_acc:
                best_acc = va
                self.model.save(best_path)
                print(f"  ✓ Best saved (val={va:.2f}%)")
        self.model.save(os.path.join(self.save_dir,"cnn_final_model.pth"))
        print(f"\nBest val accuracy: {best_acc:.2f}%")
        return history

    def _epoch(self, loader, train):
        self.model.train(train)
        total_loss, correct, total = 0.0, 0, 0
        for imgs, qs, labels in loader:
            labels = labels.to(self.device)
            if train: self.optim.zero_grad()
            logits = self.model(imgs, qs)
            loss   = self.criterion(logits, labels)
            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(
                    [p for p in self.model.parameters()
                     if p.requires_grad], 1.0)
                self.optim.step()
            total_loss += loss.item()*labels.size(0)
            correct    += (logits.argmax(-1)==labels).sum().item()
            total      += labels.size(0)
        return total_loss/max(total,1), correct/max(total,1)*100

    def plot(self, history):
        import matplotlib.pyplot as plt
        fig,(a1,a2) = plt.subplots(1,2,figsize=(12,4))
        eps = range(1,len(history["train_loss"])+1)
        a1.plot(eps,history["train_loss"],"b-o",label="Train")
        a1.plot(eps,history["val_loss"],"r-o",label="Val")
        a1.set_title("CNN Loss"); a1.legend(); a1.grid(alpha=0.3)
        a2.plot(eps,history["train_acc"],"b-o",label="Train")
        a2.plot(eps,history["val_acc"],"r-o",label="Val")
        a2.axhline(80,color="green",linestyle="--",alpha=0.6,label="80% target")
        a2.set_title("CNN Accuracy (%)"); a2.legend(); a2.grid(alpha=0.3)
        plt.tight_layout()
        path = os.path.join(self.save_dir,"cnn_training_curves.png")
        plt.savefig(path,dpi=120,bbox_inches="tight"); plt.show()
        print(f"Curves → {path}")


# ── Inference Engine ──────────────────────────────────────────────
class CNNInferenceEngine:
    """
    Reads image pixels directly through ResNet50.
    No CLIP, no embeddings — pure pixel to answer.
    """
    CHARS    = list("abcdefghijklmnopqrstuvwxyz0123456789 ?,'.")
    CHAR2IDX = {c: i+1 for i, c in enumerate(CHARS)}
    MAX_Q    = 60

    def __init__(self, model_path, answer_index_path, device="cpu"):
        self.device = device
        with open(answer_index_path) as f:
            self.answer_index = json.load(f)
        self.model = CNNVQAModel(len(self.answer_index), device)
        self.model.load(model_path)
        self.model.eval()
        self.tf = transforms.Compose([
            transforms.Resize((224,224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485,0.456,0.406],
                std= [0.229,0.224,0.225])])

    def answer(self, image: Image.Image, question: str) -> Dict:
        # Step 1: read image as pixel tensor
        img_t = self.tf(image.convert("RGB")).unsqueeze(0)
        # Step 2: tokenise question as characters
        q     = question.lower()[:self.MAX_Q]
        q_tok = [self.CHAR2IDX.get(c,0) for c in q]
        q_tok += [0]*(self.MAX_Q-len(q_tok))
        q_t   = torch.tensor([q_tok], dtype=torch.long)
        # Step 3: CNN reads pixels → answer
        answers, confs = self.model.predict(img_t, q_t, self.answer_index)
        return {"question": question, "answer": answers[0],
                "confidence": round(confs[0],4),
                "route": "cnn_pixel_reading",
                "source": "resnet50_cnn"}

    def batch_answer(self, image, questions):
        return [self.answer(image, q) for q in questions]


print("All CNN classes defined ✓")

All CNN classes defined ✓


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CNN DIRECT IMAGE READING — Cell C: Train
# ═══════════════════════════════════════════════════════════════════

CNN_MODEL_DIR  = os.path.join(BASE_DIR, "models")
CNN_BEST       = os.path.join(CNN_MODEL_DIR, "cnn_best_model.pth")
nw             = 2 if DEVICE == "cuda" else 0

with open(os.path.join(PROC_DIR,"answer_index.json")) as f:
    answer_index = json.load(f)

if os.path.exists(CNN_BEST):
    print("CNN model already trained. Loading from Drive ...")
    cnn_model = CNNVQAModel(len(answer_index), DEVICE)
    cnn_model.load(CNN_BEST)
    print("CNN model loaded ✓")
else:
    # Build datasets — raw pixel reading, no CLIP
    train_ds = CNNVQADataset(
        os.path.join(PROC_DIR,"train.json"), split="train")
    test_ds  = CNNVQADataset(
        os.path.join(PROC_DIR,"test.json"),  split="test")
    train_loader = DataLoader(
        train_ds, batch_size=32, shuffle=True,
        num_workers=nw, pin_memory=(DEVICE=="cuda"))
    test_loader  = DataLoader(
        test_ds,  batch_size=32, shuffle=False,
        num_workers=nw, pin_memory=(DEVICE=="cuda"))

    print(f"Train batches : {len(train_loader)}")
    print(f"Test  batches : {len(test_loader)}")
    print(f"Classes       : {len(answer_index)}")

    cnn_model = CNNVQAModel(
        num_classes=len(answer_index), device=DEVICE, feat_dim=512)
    tp = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
    print(f"Trainable params: {tp:,}")

    trainer = CNNVQATrainer(
        model=cnn_model, train_loader=train_loader,
        val_loader=test_loader, device=DEVICE,
        lr=1e-4, epochs=15, save_dir=CNN_MODEL_DIR)

    cnn_history = trainer.train()
    best = max(cnn_history["val_acc"])
    print(f"\nBest CNN val accuracy: {best:.2f}%")
    trainer.plot(cnn_history)

CNN model already trained. Loading from Drive ...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 88.8MB/s]


CNN model loaded ✓


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CNN LIVE PILOT DEMO — Image is READ via ResNet50 CNN pixel processing
# Paste this AFTER Cell D (CNN Inference & Test)
#             BEFORE Cell E (Compare CNN vs Hybrid)
# ═══════════════════════════════════════════════════════════════════

# ── All imports self-contained — no errors even after restart ─────
import os, sys, json, random, io
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
from torchvision import transforms, models
from datetime import datetime
from typing import Dict, List

# ── Ensure project is on path ─────────────────────────────────────
PROJECT = "/content/aviation_vqa"
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

# ── Ensure BASE_DIR is set ────────────────────────────────────────
if "BASE_DIR" not in dir():
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/aviation_vqa_output"

PROC_DIR = os.path.join(BASE_DIR, "data/processed")
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

# ── Re-define CNN classes (safe even if already defined) ──────────
class _CNNVisualEncoder(nn.Module):
    def __init__(self, out_dim=512):
        super().__init__()
        backbone      = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        self.proj     = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, out_dim),
            nn.LayerNorm(out_dim),
            nn.GELU(),
            nn.Dropout(0.15))
    def forward(self, x):
        return self.proj(self.features(x))

class _QuestionEncoder(nn.Module):
    def __init__(self, vocab=40, embed=64, hidden=256, out_dim=512):
        super().__init__()
        self.embed = nn.Embedding(vocab+1, embed, padding_idx=0)
        self.gru   = nn.GRU(embed, hidden, num_layers=2,
                            batch_first=True, dropout=0.2,
                            bidirectional=True)
        self.proj  = nn.Sequential(
            nn.Linear(hidden*2, out_dim),
            nn.LayerNorm(out_dim), nn.GELU(), nn.Dropout(0.15))
    def forward(self, x):
        emb = self.embed(x)
        _, h = self.gru(emb)
        return self.proj(torch.cat([h[-2], h[-1]], dim=-1))

class _CNNVQAModel(nn.Module):
    def __init__(self, num_classes, device="cpu", feat_dim=512):
        super().__init__()
        self.device       = device
        self.visual_enc   = _CNNVisualEncoder(out_dim=feat_dim)
        self.question_enc = _QuestionEncoder(out_dim=feat_dim)
        self.cross_attn   = nn.MultiheadAttention(
            feat_dim, num_heads=8, dropout=0.1, batch_first=True)
        self.norm1        = nn.LayerNorm(feat_dim)
        self.classifier   = nn.Sequential(
            nn.Linear(feat_dim*2, feat_dim), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(feat_dim, feat_dim//2), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(feat_dim//2, num_classes))
        self.to(device)

    def forward(self, images, questions):
        images    = images.to(self.device)
        questions = questions.to(self.device)
        v = self.visual_enc(images)
        q = self.question_enc(questions)
        v_s = v.unsqueeze(1)
        q_s = q.unsqueeze(1)
        att, _ = self.cross_attn(v_s, q_s, q_s)
        fused   = self.norm1(v_s + att).squeeze(1)
        return self.classifier(torch.cat([fused, q], dim=-1))

    def predict(self, images, questions, answer_index):
        self.eval()
        with torch.no_grad():
            logits = self.forward(images, questions)
            probs  = F.softmax(logits, dim=-1)
            ids    = logits.argmax(-1).cpu().tolist()
            confs  = probs.max(-1).values.cpu().tolist()
        idx2ans = {v: k for k, v in answer_index.items()}
        return [idx2ans.get(i, "unknown") for i in ids], confs

    def load(self, path):
        self.load_state_dict(
            torch.load(path, map_location=self.device,
                       weights_only=True))
        return self

# ── CNN Image Reader — core class ─────────────────────────────────
class CNNImageReader:
    """
    Reads uploaded image pixels directly through ResNet50 CNN.
    No CLIP, no embeddings.
    Step 1: PIL Image → resize to 224x224
    Step 2: Convert to float tensor → normalise pixels
    Step 3: ResNet50 reads pixel patterns (edges, blips, density)
    Step 4: Cross-attention with question → answer
    """
    CHARS    = list("abcdefghijklmnopqrstuvwxyz0123456789 ?,'.")
    CHAR2IDX = {c: i+1 for i, c in enumerate(CHARS)}
    MAX_Q    = 60

    def __init__(self, model_path: str, answer_index: dict, device: str):
        self.device       = device
        self.answer_index = answer_index
        self.model        = _CNNVQAModel(len(answer_index), device)
        self.model.load(model_path)
        self.model.eval()

        # Pixel transform — reads raw RGB values
        self.pixel_tf = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),               # pixels → [0,1] tensor
            transforms.Normalize(                # ImageNet normalisation
                mean=[0.485, 0.456, 0.406],
                std= [0.229, 0.224, 0.225]),
        ])
        print(f"CNN Image Reader loaded on {device} ✓")
        print("Pipeline: uploaded pixels → ResNet50 → answer")

    def read_and_answer(self, image: Image.Image,
                        question: str) -> Dict:
        """
        STEP 1: Read raw pixels from uploaded image
        STEP 2: CNN processes pixel patterns
        STEP 3: Combine with question features
        STEP 4: Return answer
        """
        # Step 1 — read pixels
        img_array = np.array(image.convert("RGB").resize((224, 224)))

        # Step 2 — convert to normalised tensor (what CNN reads)
        img_tensor = self.pixel_tf(
            image.convert("RGB")).unsqueeze(0)  # (1,3,224,224)

        # Step 3 — tokenise question as characters
        q     = question.lower()[:self.MAX_Q]
        q_tok = [self.CHAR2IDX.get(c, 0) for c in q]
        q_tok += [0] * (self.MAX_Q - len(q_tok))
        q_tensor = torch.tensor([q_tok], dtype=torch.long)

        # Step 4 — CNN reads pixels and answers
        answers, confs = self.model.predict(
            img_tensor, q_tensor, self.answer_index)

        return {
            "question":     question,
            "answer":       answers[0],
            "confidence":   round(confs[0], 4),
            "route":        "CNN pixel reading",
            "source":       "ResNet50",
            "pixel_shape":  str(img_array.shape),
            "pixel_min":    int(img_array.min()),
            "pixel_max":    int(img_array.max()),
        }


# ── Load CNN model ────────────────────────────────────────────────
CNN_BEST = os.path.join(BASE_DIR, "models/cnn_best_model.pth")

with open(os.path.join(PROC_DIR, "answer_index.json")) as f:
    answer_index = json.load(f)

if not os.path.exists(CNN_BEST):
    raise FileNotFoundError(
        f"CNN model not found at {CNN_BEST}\n"
        "Run Cell C (CNN Training) first.")

cnn_reader = CNNImageReader(
    model_path   = CNN_BEST,
    answer_index = answer_index,
    device       = DEVICE)

print(f"\nAnswer classes: {len(answer_index)}")
print("CNN Image Reader ready ✓")


# ══════════════════════════════════════════════════════════════════
# LIVE DEMO GUI — CNN reads uploaded image pixels
# ══════════════════════════════════════════════════════════════════

IMAGE_DIR = os.path.join(PROC_DIR, "images")

PRESET_QUESTIONS = [
    "How many aircraft are visible on the radar?",
    "Is there any aircraft on the left side of the radar?",
    "Is there any aircraft on the right side of the radar?",
    "Is there any aircraft in the upper portion of the radar?",
    "Is there any aircraft in the lower portion of the radar?",
    "Is there any aircraft near the center of the radar?",
    "Are there more than 4 aircraft on the radar?",
    "What is the callsign of the aircraft with the highest altitude?",
    "What is the callsign of the aircraft with the lowest altitude?",
    "Which quadrant of the radar has the most aircraft?",
    "How many aircraft are flying above 30,000 feet?",
    "What is the approximate altitude of the highest aircraft in thousands of feet?",
]

# ── State ─────────────────────────────────────────────────────────
current_image  = {"pil": None, "path": None}
session_log    = []

# ── Widgets ───────────────────────────────────────────────────────
header = widgets.HTML("""
<div style="background:linear-gradient(135deg,#0a1628,#1a3a5c);
            padding:16px 20px;border-radius:10px;margin-bottom:10px">
  <h2 style="color:#00d4ff;margin:0;font-family:monospace;
             letter-spacing:2px">
    ✈ VISUAL CO-PILOT — CNN PIXEL READER</h2>
  <p style="color:#8ab4d4;margin:4px 0 0 0;font-size:12px">
    Image pixels read directly by ResNet50 CNN — no embeddings</p>
</div>""")

# Image display
img_out = widgets.Output(layout=widgets.Layout(
    width="300px", height="300px",
    border="2px solid #00d4ff", border_radius="8px"))

# Pixel info display
pixel_info = widgets.HTML(
    value="<span style='color:#8ab4d4;font-size:11px'>"
          "No image loaded</span>")

# Buttons
btn_random = widgets.Button(
    description="🎲 Random Image", button_style="info",
    layout=widgets.Layout(width="150px"))
btn_upload = widgets.Button(
    description="📁 Upload Image", button_style="warning",
    layout=widgets.Layout(width="150px"))
upload_widget = widgets.FileUpload(
    accept=".png,.jpg,.jpeg", multiple=False,
    layout=widgets.Layout(display="none"))

# Question input
q_input = widgets.Text(
    placeholder="Type your question here and press Enter …",
    layout=widgets.Layout(width="100%"),
    style={"description_width": "0px"})

btn_ask = widgets.Button(
    description="🛩 Ask CNN",
    button_style="success",
    layout=widgets.Layout(width="110px", height="36px"))

# Answer display
answer_out = widgets.Output(layout=widgets.Layout(
    min_height="90px",
    border="1px solid #1a3a5c",
    border_radius="8px",
    padding="8px", margin="6px 0"))

# History
history_out = widgets.Output(layout=widgets.Layout(
    height="200px", overflow_y="auto",
    border="1px solid #1a3a5c",
    border_radius="8px", padding="8px"))

btn_clear = widgets.Button(
    description="🗑 Clear",
    button_style="danger",
    layout=widgets.Layout(width="100px"))

status = widgets.HTML(
    value="<span style='color:#8ab4d4'>Ready — load an image to begin</span>")

# ── Helper: load image ────────────────────────────────────────────
def load_image(path_or_bytes, label=""):
    try:
        if isinstance(path_or_bytes, str):
            img = Image.open(path_or_bytes).convert("RGB")
            current_image["path"] = path_or_bytes
        else:
            img = Image.open(io.BytesIO(path_or_bytes)).convert("RGB")
            current_image["path"] = "uploaded"

        current_image["pil"] = img
        arr = np.array(img.resize((224, 224)))

        with img_out:
            clear_output(wait=True)
            display(img.resize((280, 280)))

        pixel_info.value = (
            f"<span style='color:#8ab4d4;font-size:11px'>"
            f"📍 {label or 'image loaded'} | "
            f"pixels: {arr.shape} | "
            f"min={arr.min()} max={arr.max()} "
            f"mean={arr.mean():.1f}</span>")

        set_status(f"Image loaded: {label} ✓")
    except Exception as e:
        set_status(f"Error loading image: {e}", "red")

# ── Helper: ask question ──────────────────────────────────────────
def ask_question(question: str):
    if current_image["pil"] is None:
        set_status("⚠️ Load an image first", "orange")
        return
    if not question.strip():
        set_status("⚠️ Type a question first", "orange")
        return

    set_status("CNN reading image pixels …")
    try:
        result = cnn_reader.read_and_answer(
            current_image["pil"], question)
        show_answer(question, result)
        add_history(question, result)
        set_status("Ready ✓")
    except Exception as e:
        set_status(f"Error: {e}", "red")

# ── Helper: show answer ───────────────────────────────────────────
def show_answer(question: str, result: Dict):
    ans   = result["answer"]
    conf  = result["confidence"]
    src   = result["source"]
    pshp  = result["pixel_shape"]
    pmin  = result["pixel_min"]
    pmax  = result["pixel_max"]
    conf_pct   = int(conf * 100)
    conf_color = ("#00b894" if conf_pct >= 70
                  else "#fdcb6e" if conf_pct >= 40
                  else "#d63031")

    html = f"""
    <div style="font-family:monospace;padding:6px">
      <div style="color:#8ab4d4;font-size:11px;margin-bottom:4px">
        Q: {question}</div>
      <div style="display:flex;align-items:center;gap:10px;margin-bottom:6px">
        <span style="background:#6c5ce7;color:white;padding:2px 8px;
                     border-radius:4px;font-size:11px;font-weight:bold">
          CNN PIXEL READ</span>
        <span style="color:white;font-size:20px;font-weight:bold">
          {ans}</span>
      </div>
      <div style="background:#1a3a5c;border-radius:4px;
                  height:8px;width:100%;margin-bottom:4px">
        <div style="background:{conf_color};width:{conf_pct}%;
                    height:8px;border-radius:4px"></div>
      </div>
      <div style="color:#8ab4d4;font-size:10px">
        Confidence: {conf_pct}% &nbsp;|&nbsp;
        Model: {src} &nbsp;|&nbsp;
        Pixel shape: {pshp} &nbsp;|&nbsp;
        Range: [{pmin}, {pmax}]
      </div>
    </div>"""

    with answer_out:
        clear_output(wait=True)
        display(widgets.HTML(html))

# ── Helper: add to history ────────────────────────────────────────
def add_history(question: str, result: Dict):
    ts  = datetime.now().strftime("%H:%M:%S")
    ans = result["answer"]
    conf= int(result["confidence"] * 100)
    session_log.append({
        "time": ts, "question": question,
        "answer": ans, "confidence": conf})
    row = f"""
    <div style="border-bottom:1px solid #1a3a5c;
                padding:3px 0;font-family:monospace">
      <span style="color:#636e72;font-size:10px">{ts}</span>
      <span style="background:#6c5ce7;color:white;padding:1px 5px;
                   border-radius:3px;font-size:10px;margin:0 5px">CNN</span>
      <span style="color:#8ab4d4;font-size:11px">
        {question[:45]}</span><br>
      <span style="color:white;font-size:12px;padding-left:80px">
        → {ans}
        <span style="color:#636e72"> ({conf}%)</span>
      </span>
    </div>"""
    with history_out:
        display(widgets.HTML(row))

def set_status(msg, color="#8ab4d4"):
    status.value = (f"<span style='color:{color};"
                    f"font-size:12px'>{msg}</span>")

# ── Event handlers ────────────────────────────────────────────────
def on_random(_):
    try:
        files_list = [f for f in os.listdir(IMAGE_DIR)
                      if f.endswith((".png",".jpg",".jpeg"))]
        if not files_list:
            set_status("No images found in image dir", "orange")
            return
        fname = random.choice(files_list)
        load_image(os.path.join(IMAGE_DIR, fname), fname)
    except Exception as e:
        set_status(f"Error: {e}", "red")

def on_upload_click(_):
    display(upload_widget)

def on_file_upload(change):
    if not change["new"]: return
    name    = list(change["new"].keys())[0]
    content = change["new"][name]["content"]
    load_image(content, name)

def on_ask(_):
    ask_question(q_input.value.strip())
    q_input.value = ""

def on_clear(_):
    session_log.clear()
    with history_out: clear_output()
    with answer_out:  clear_output()
    set_status("Cleared ✓")

btn_random.on_click(on_random)
btn_upload.on_click(on_upload_click)
upload_widget.observe(on_file_upload, names="value")
btn_ask.on_click(on_ask)
q_input.on_submit(on_ask)
btn_clear.on_click(on_clear)

# ── Preset question buttons ───────────────────────────────────────
preset_btns = []
for q in PRESET_QUESTIONS:
    short = q[:48] + "…" if len(q) > 48 else q
    b = widgets.Button(
        description=short, tooltip=q,
        layout=widgets.Layout(width="auto", height="28px", margin="2px"))
    b.style.button_color = "#1a3a5c"
    b.on_click(lambda ev, question=q: ask_question(question))
    preset_btns.append(b)

rows = []
for i in range(0, len(preset_btns), 2):
    rows.append(widgets.HBox(preset_btns[i:i+2]))
preset_grid = widgets.VBox(rows)

# ── Layout ────────────────────────────────────────────────────────
left = widgets.VBox([
    widgets.HTML("<b style='color:#00d4ff'>Radar Image</b>"),
    img_out,
    widgets.HBox([btn_random, btn_upload]),
    upload_widget,
    pixel_info,
], layout=widgets.Layout(width="320px", margin="0 16px 0 0"))

right = widgets.VBox([
    widgets.HTML("<b style='color:#00d4ff'>Ask the CNN Co-Pilot</b>"),
    widgets.HBox([q_input, btn_ask]),
    widgets.HTML("<p style='color:#8ab4d4;font-size:11px;"
                 "margin:6px 0 3px'>Quick questions:</p>"),
    preset_grid,
    widgets.HTML("<b style='color:#00d4ff;margin-top:6px'>Answer</b>"),
    answer_out,
], layout=widgets.Layout(flex="1"))

root = widgets.VBox([
    header,
    widgets.HBox([left, right],
                 layout=widgets.Layout(align_items="flex-start")),
    widgets.HTML("<hr style='border-color:#1a3a5c;margin:10px 0'>"),
    widgets.HTML("<b style='color:#00d4ff'>Session History</b>"),
    history_out,
    widgets.HBox([btn_clear]),
    status,
], layout=widgets.Layout(padding="14px"))

display(root)

# Auto-load a random image on start
on_random(None)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 113MB/s]


CNN Image Reader loaded on cuda ✓
Pipeline: uploaded pixels → ResNet50 → answer

Answer classes: 2119
CNN Image Reader ready ✓


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CNN DIRECT IMAGE READING — Cell D: Inference & Test
# ═══════════════════════════════════════════════════════════════════
import random

# Load CNN inference engine
cnn_engine = CNNInferenceEngine(
    model_path        = CNN_BEST,
    answer_index_path = os.path.join(PROC_DIR,"answer_index.json"),
    device            = DEVICE)

print("CNN Inference Engine loaded ✓")
print("Pipeline: raw pixels → ResNet50 CNN → answer (no embeddings)\n")

# Pick a random test image
img_files   = os.listdir(os.path.join(PROC_DIR,"images"))
sample_path = os.path.join(PROC_DIR,"images", random.choice(img_files))
img         = Image.open(sample_path)

# Show the image
import matplotlib.pyplot as plt
plt.figure(figsize=(4,4))
plt.imshow(img); plt.title(f"Test: {os.path.basename(sample_path)}")
plt.axis("off"); plt.show()

# Test questions
questions = [
    "How many aircraft are visible on the radar?",
    "Is there any aircraft on the left side of the radar?",
    "Is there any aircraft in the upper portion of the radar?",
    "Are there more than 4 aircraft on the radar?",
    "What is the callsign of the aircraft with the highest altitude?",
    "Which quadrant of the radar has the most aircraft?",
    "How many aircraft are flying above 30,000 feet?",
    "Is there any aircraft near the center of the radar?",
]

print(f"{'QUESTION':<55} {'ANSWER':<22} CONF")
print("-"*90)
for q in questions:
    r = cnn_engine.answer(img, q)
    print(f"{q:<55} {r['answer']:<22} {r['confidence']:.3f}")

print(f"\nSource: {r['source']} | Route: {r['route']}")

CNN Inference Engine loaded ✓
Pipeline: raw pixels → ResNet50 CNN → answer (no embeddings)

QUESTION                                                ANSWER                 CONF
------------------------------------------------------------------------------------------
How many aircraft are visible on the radar?             8                      0.656
Is there any aircraft on the left side of the radar?    yes                    0.971
Is there any aircraft in the upper portion of the radar? yes                    0.968
Are there more than 4 aircraft on the radar?            yes                    0.975
What is the callsign of the aircraft with the highest altitude? yes                    0.007
Which quadrant of the radar has the most aircraft?      top-left               0.589
How many aircraft are flying above 30,000 feet?         5                      0.159
Is there any aircraft near the center of the radar?     yes                    0.980

Source: resnet50_cnn | Route: cnn_pixel_rea

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CNN LIVE PILOT DEMO — Image is READ via ResNet50 CNN pixel processing
# Paste this AFTER Cell D (CNN Inference & Test)
#             BEFORE Cell E (Compare CNN vs Hybrid)
# ═══════════════════════════════════════════════════════════════════

# ── All imports self-contained — no errors even after restart ─────
import os, sys, json, random, io
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
from torchvision import transforms, models
from datetime import datetime
from typing import Dict, List

# ── Ensure project is on path ─────────────────────────────────────
PROJECT = "/content/aviation_vqa"
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

# ── Ensure BASE_DIR is set ────────────────────────────────────────
if "BASE_DIR" not in dir():
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/aviation_vqa_output"

PROC_DIR = os.path.join(BASE_DIR, "data/processed")
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

# ── Re-define CNN classes (safe even if already defined) ──────────
class _CNNVisualEncoder(nn.Module):
    def __init__(self, out_dim=512):
        super().__init__()
        backbone      = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        self.proj     = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, out_dim),
            nn.LayerNorm(out_dim),
            nn.GELU(),
            nn.Dropout(0.15))
    def forward(self, x):
        return self.proj(self.features(x))

class _QuestionEncoder(nn.Module):
    def __init__(self, vocab=40, embed=64, hidden=256, out_dim=512):
        super().__init__()
        self.embed = nn.Embedding(vocab+1, embed, padding_idx=0)
        self.gru   = nn.GRU(embed, hidden, num_layers=2,
                            batch_first=True, dropout=0.2,
                            bidirectional=True)
        self.proj  = nn.Sequential(
            nn.Linear(hidden*2, out_dim),
            nn.LayerNorm(out_dim), nn.GELU(), nn.Dropout(0.15))
    def forward(self, x):
        emb = self.embed(x)
        _, h = self.gru(emb)
        return self.proj(torch.cat([h[-2], h[-1]], dim=-1))

class _CNNVQAModel(nn.Module):
    def __init__(self, num_classes, device="cpu", feat_dim=512):
        super().__init__()
        self.device       = device
        self.visual_enc   = _CNNVisualEncoder(out_dim=feat_dim)
        self.question_enc = _QuestionEncoder(out_dim=feat_dim)
        self.cross_attn   = nn.MultiheadAttention(
            feat_dim, num_heads=8, dropout=0.1, batch_first=True)
        self.norm1        = nn.LayerNorm(feat_dim)
        self.classifier   = nn.Sequential(
            nn.Linear(feat_dim*2, feat_dim), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(feat_dim, feat_dim//2), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(feat_dim//2, num_classes))
        self.to(device)

    def forward(self, images, questions):
        images    = images.to(self.device)
        questions = questions.to(self.device)
        v = self.visual_enc(images)
        q = self.question_enc(questions)
        v_s = v.unsqueeze(1)
        q_s = q.unsqueeze(1)
        att, _ = self.cross_attn(v_s, q_s, q_s)
        fused   = self.norm1(v_s + att).squeeze(1)
        return self.classifier(torch.cat([fused, q], dim=-1))

    def predict(self, images, questions, answer_index):
        self.eval()
        with torch.no_grad():
            logits = self.forward(images, questions)
            probs  = F.softmax(logits, dim=-1)
            ids    = logits.argmax(-1).cpu().tolist()
            confs  = probs.max(-1).values.cpu().tolist()
        idx2ans = {v: k for k, v in answer_index.items()}
        return [idx2ans.get(i, "unknown") for i in ids], confs

    def load(self, path):
        self.load_state_dict(
            torch.load(path, map_location=self.device,
                       weights_only=True))
        return self

# ── CNN Image Reader — core class ─────────────────────────────────
class CNNImageReader:
    """
    Reads uploaded image pixels directly through ResNet50 CNN.
    No CLIP, no embeddings.
    Step 1: PIL Image → resize to 224x224
    Step 2: Convert to float tensor → normalise pixels
    Step 3: ResNet50 reads pixel patterns (edges, blips, density)
    Step 4: Cross-attention with question → answer
    """
    CHARS    = list("abcdefghijklmnopqrstuvwxyz0123456789 ?,'.")
    CHAR2IDX = {c: i+1 for i, c in enumerate(CHARS)}
    MAX_Q    = 60

    def __init__(self, model_path: str, answer_index: dict, device: str):
        self.device       = device
        self.answer_index = answer_index
        self.model        = _CNNVQAModel(len(answer_index), device)
        self.model.load(model_path)
        self.model.eval()

        # Pixel transform — reads raw RGB values
        self.pixel_tf = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),               # pixels → [0,1] tensor
            transforms.Normalize(                # ImageNet normalisation
                mean=[0.485, 0.456, 0.406],
                std= [0.229, 0.224, 0.225]),
        ])
        print(f"CNN Image Reader loaded on {device} ✓")
        print("Pipeline: uploaded pixels → ResNet50 → answer")

    def read_and_answer(self, image: Image.Image,
                        question: str) -> Dict:
        """
        STEP 1: Read raw pixels from uploaded image
        STEP 2: CNN processes pixel patterns
        STEP 3: Combine with question features
        STEP 4: Return answer
        """
        # Step 1 — read pixels
        img_array = np.array(image.convert("RGB").resize((224, 224)))

        # Step 2 — convert to normalised tensor (what CNN reads)
        img_tensor = self.pixel_tf(
            image.convert("RGB")).unsqueeze(0)  # (1,3,224,224)

        # Step 3 — tokenise question as characters
        q     = question.lower()[:self.MAX_Q]
        q_tok = [self.CHAR2IDX.get(c, 0) for c in q]
        q_tok += [0] * (self.MAX_Q - len(q_tok))
        q_tensor = torch.tensor([q_tok], dtype=torch.long)

        # Step 4 — CNN reads pixels and answers
        answers, confs = self.model.predict(
            img_tensor, q_tensor, self.answer_index)

        return {
            "question":     question,
            "answer":       answers[0],
            "confidence":   round(confs[0], 4),
            "route":        "CNN pixel reading",
            "source":       "ResNet50",
            "pixel_shape":  str(img_array.shape),
            "pixel_min":    int(img_array.min()),
            "pixel_max":    int(img_array.max()),
        }


# ── Load CNN model ────────────────────────────────────────────────
CNN_BEST = os.path.join(BASE_DIR, "models/cnn_best_model.pth")

with open(os.path.join(PROC_DIR, "answer_index.json")) as f:
    answer_index = json.load(f)

if not os.path.exists(CNN_BEST):
    raise FileNotFoundError(
        f"CNN model not found at {CNN_BEST}\n"
        "Run Cell C (CNN Training) first.")

cnn_reader = CNNImageReader(
    model_path   = CNN_BEST,
    answer_index = answer_index,
    device       = DEVICE)

print(f"\nAnswer classes: {len(answer_index)}")
print("CNN Image Reader ready ✓")


# ══════════════════════════════════════════════════════════════════
# LIVE DEMO GUI — CNN reads uploaded image pixels
# ══════════════════════════════════════════════════════════════════

IMAGE_DIR = os.path.join(PROC_DIR, "images")

PRESET_QUESTIONS = [
    "How many aircraft are visible on the radar?",
    "Is there any aircraft on the left side of the radar?",
    "Is there any aircraft on the right side of the radar?",
    "Is there any aircraft in the upper portion of the radar?",
    "Is there any aircraft in the lower portion of the radar?",
    "Is there any aircraft near the center of the radar?",
    "Are there more than 4 aircraft on the radar?",
    "What is the callsign of the aircraft with the highest altitude?",
    "What is the callsign of the aircraft with the lowest altitude?",
    "Which quadrant of the radar has the most aircraft?",
    "How many aircraft are flying above 30,000 feet?",
    "What is the approximate altitude of the highest aircraft in thousands of feet?",
]

# ── State ─────────────────────────────────────────────────────────
current_image  = {"pil": None, "path": None}
session_log    = []

# ── Widgets ───────────────────────────────────────────────────────
header = widgets.HTML("""
<div style="background:linear-gradient(135deg,#0a1628,#1a3a5c);
            padding:16px 20px;border-radius:10px;margin-bottom:10px">
  <h2 style="color:#00d4ff;margin:0;font-family:monospace;
             letter-spacing:2px">
    ✈ VISUAL CO-PILOT — CNN PIXEL READER</h2>
  <p style="color:#8ab4d4;margin:4px 0 0 0;font-size:12px">
    Image pixels read directly by ResNet50 CNN — no embeddings</p>
</div>""")

# Image display
img_out = widgets.Output(layout=widgets.Layout(
    width="300px", height="300px",
    border="2px solid #00d4ff", border_radius="8px"))

# Pixel info display
pixel_info = widgets.HTML(
    value="<span style='color:#8ab4d4;font-size:11px'>"
          "No image loaded</span>")

# Buttons
btn_random = widgets.Button(
    description="🎲 Random Image", button_style="info",
    layout=widgets.Layout(width="150px"))
btn_upload = widgets.Button(
    description="📁 Upload Image", button_style="warning",
    layout=widgets.Layout(width="150px"))
upload_widget = widgets.FileUpload(
    accept=".png,.jpg,.jpeg", multiple=False,
    layout=widgets.Layout(display="none"))

# Question input
q_input = widgets.Text(
    placeholder="Type your question here and press Enter …",
    layout=widgets.Layout(width="100%"),
    style={"description_width": "0px"})

btn_ask = widgets.Button(
    description="🛩 Ask CNN",
    button_style="success",
    layout=widgets.Layout(width="110px", height="36px"))

# Answer display
answer_out = widgets.Output(layout=widgets.Layout(
    min_height="90px",
    border="1px solid #1a3a5c",
    border_radius="8px",
    padding="8px", margin="6px 0"))

# History
history_out = widgets.Output(layout=widgets.Layout(
    height="200px", overflow_y="auto",
    border="1px solid #1a3a5c",
    border_radius="8px", padding="8px"))

btn_clear = widgets.Button(
    description="🗑 Clear",
    button_style="danger",
    layout=widgets.Layout(width="100px"))

status = widgets.HTML(
    value="<span style='color:#8ab4d4'>Ready — load an image to begin</span>")

# ── Helper: load image ────────────────────────────────────────────
def load_image(path_or_bytes, label=""):
    try:
        if isinstance(path_or_bytes, str):
            img = Image.open(path_or_bytes).convert("RGB")
            current_image["path"] = path_or_bytes
        else:
            img = Image.open(io.BytesIO(path_or_bytes)).convert("RGB")
            current_image["path"] = "uploaded"

        current_image["pil"] = img
        arr = np.array(img.resize((224, 224)))

        with img_out:
            clear_output(wait=True)
            display(img.resize((280, 280)))

        pixel_info.value = (
            f"<span style='color:#8ab4d4;font-size:11px'>"
            f"📍 {label or 'image loaded'} | "
            f"pixels: {arr.shape} | "
            f"min={arr.min()} max={arr.max()} "
            f"mean={arr.mean():.1f}</span>")

        set_status(f"Image loaded: {label} ✓")
    except Exception as e:
        set_status(f"Error loading image: {e}", "red")

# ── Helper: ask question ──────────────────────────────────────────
def ask_question(question: str):
    if current_image["pil"] is None:
        set_status("⚠️ Load an image first", "orange")
        return
    if not question.strip():
        set_status("⚠️ Type a question first", "orange")
        return

    set_status("CNN reading image pixels …")
    try:
        result = cnn_reader.read_and_answer(
            current_image["pil"], question)
        show_answer(question, result)
        add_history(question, result)
        set_status("Ready ✓")
    except Exception as e:
        set_status(f"Error: {e}", "red")

# ── Helper: show answer ───────────────────────────────────────────
def show_answer(question: str, result: Dict):
    ans   = result["answer"]
    conf  = result["confidence"]
    src   = result["source"]
    pshp  = result["pixel_shape"]
    pmin  = result["pixel_min"]
    pmax  = result["pixel_max"]
    conf_pct   = int(conf * 100)
    conf_color = ("#00b894" if conf_pct >= 70
                  else "#fdcb6e" if conf_pct >= 40
                  else "#d63031")

    html = f"""
    <div style="font-family:monospace;padding:6px">
      <div style="color:#8ab4d4;font-size:11px;margin-bottom:4px">
        Q: {question}</div>
      <div style="display:flex;align-items:center;gap:10px;margin-bottom:6px">
        <span style="background:#6c5ce7;color:white;padding:2px 8px;
                     border-radius:4px;font-size:11px;font-weight:bold">
          CNN PIXEL READ</span>
        <span style="color:white;font-size:20px;font-weight:bold">
          {ans}</span>
      </div>
      <div style="background:#1a3a5c;border-radius:4px;
                  height:8px;width:100%;margin-bottom:4px">
        <div style="background:{conf_color};width:{conf_pct}%;
                    height:8px;border-radius:4px"></div>
      </div>
      <div style="color:#8ab4d4;font-size:10px">
        Confidence: {conf_pct}% &nbsp;|&nbsp;
        Model: {src} &nbsp;|&nbsp;
        Pixel shape: {pshp} &nbsp;|&nbsp;
        Range: [{pmin}, {pmax}]
      </div>
    </div>"""

    with answer_out:
        clear_output(wait=True)
        display(widgets.HTML(html))

# ── Helper: add to history ────────────────────────────────────────
def add_history(question: str, result: Dict):
    ts  = datetime.now().strftime("%H:%M:%S")
    ans = result["answer"]
    conf= int(result["confidence"] * 100)
    session_log.append({
        "time": ts, "question": question,
        "answer": ans, "confidence": conf})
    row = f"""
    <div style="border-bottom:1px solid #1a3a5c;
                padding:3px 0;font-family:monospace">
      <span style="color:#636e72;font-size:10px">{ts}</span>
      <span style="background:#6c5ce7;color:white;padding:1px 5px;
                   border-radius:3px;font-size:10px;margin:0 5px">CNN</span>
      <span style="color:#8ab4d4;font-size:11px">
        {question[:45]}</span><br>
      <span style="color:white;font-size:12px;padding-left:80px">
        → {ans}
        <span style="color:#636e72"> ({conf}%)</span>
      </span>
    </div>"""
    with history_out:
        display(widgets.HTML(row))

def set_status(msg, color="#8ab4d4"):
    status.value = (f"<span style='color:{color};"
                    f"font-size:12px'>{msg}</span>")

# ── Event handlers ────────────────────────────────────────────────
def on_random(_):
    try:
        files_list = [f for f in os.listdir(IMAGE_DIR)
                      if f.endswith((".png",".jpg",".jpeg"))]
        if not files_list:
            set_status("No images found in image dir", "orange")
            return
        fname = random.choice(files_list)
        load_image(os.path.join(IMAGE_DIR, fname), fname)
    except Exception as e:
        set_status(f"Error: {e}", "red")

def on_upload_click(_):
    display(upload_widget)

def on_file_upload(change):
    if not change["new"]: return
    name    = list(change["new"].keys())[0]
    content = change["new"][name]["content"]
    load_image(content, name)

def on_ask(_):
    ask_question(q_input.value.strip())
    q_input.value = ""

def on_clear(_):
    session_log.clear()
    with history_out: clear_output()
    with answer_out:  clear_output()
    set_status("Cleared ✓")

btn_random.on_click(on_random)
btn_upload.on_click(on_upload_click)
upload_widget.observe(on_file_upload, names="value")
btn_ask.on_click(on_ask)
q_input.on_submit(on_ask)
btn_clear.on_click(on_clear)

# ── Preset question buttons ───────────────────────────────────────
preset_btns = []
for q in PRESET_QUESTIONS:
    short = q[:48] + "…" if len(q) > 48 else q
    b = widgets.Button(
        description=short, tooltip=q,
        layout=widgets.Layout(width="auto", height="28px", margin="2px"))
    b.style.button_color = "#1a3a5c"
    b.on_click(lambda ev, question=q: ask_question(question))
    preset_btns.append(b)

rows = []
for i in range(0, len(preset_btns), 2):
    rows.append(widgets.HBox(preset_btns[i:i+2]))
preset_grid = widgets.VBox(rows)

# ── Layout ────────────────────────────────────────────────────────
left = widgets.VBox([
    widgets.HTML("<b style='color:#00d4ff'>Radar Image</b>"),
    img_out,
    widgets.HBox([btn_random, btn_upload]),
    upload_widget,
    pixel_info,
], layout=widgets.Layout(width="320px", margin="0 16px 0 0"))

right = widgets.VBox([
    widgets.HTML("<b style='color:#00d4ff'>Ask the CNN Co-Pilot</b>"),
    widgets.HBox([q_input, btn_ask]),
    widgets.HTML("<p style='color:#8ab4d4;font-size:11px;"
                 "margin:6px 0 3px'>Quick questions:</p>"),
    preset_grid,
    widgets.HTML("<b style='color:#00d4ff;margin-top:6px'>Answer</b>"),
    answer_out,
], layout=widgets.Layout(flex="1"))

root = widgets.VBox([
    header,
    widgets.HBox([left, right],
                 layout=widgets.Layout(align_items="flex-start")),
    widgets.HTML("<hr style='border-color:#1a3a5c;margin:10px 0'>"),
    widgets.HTML("<b style='color:#00d4ff'>Session History</b>"),
    history_out,
    widgets.HBox([btn_clear]),
    status,
], layout=widgets.Layout(padding="14px"))

display(root)

# Auto-load a random image on start
on_random(None)

CNN Image Reader loaded on cuda ✓
Pipeline: uploaded pixels → ResNet50 → answer

Answer classes: 2119
CNN Image Reader ready ✓


## 📊 Step 11 — Evaluation

In [ ]:
import os, sys, json
sys.path.insert(0, '/content/aviation_vqa')
from evaluation.evaluator import VQAEvaluator

assert "cnn_engine" in dir(), "Run the CNN inference cell first so `cnn_engine` (ResNet50) exists."

EVAL_DIR = os.path.join(BASE_DIR, 'evaluation_results')
PROC_DIR = os.path.join(BASE_DIR, 'data/processed')

with open(os.path.join(PROC_DIR,'answer_index.json')) as f:
    answer_index = json.load(f)

# Evaluates cnn_engine (ResNet50 + GRU question encoder) — no CLIP involved.
evaluator = VQAEvaluator(
    engine=cnn_engine,
    test_json=os.path.join(PROC_DIR,'test.json'),
    image_dir=os.path.join(PROC_DIR,'images'),
    answer_index=answer_index,
    output_dir=EVAL_DIR)

metrics = evaluator.run()
print('\n========== EVALUATION RESULTS (ResNet50 CNN) ==========')
print(f'  Overall Accuracy   : {metrics["overall_acc"]:>6.2f}%')
print(f'  Numerical Q Acc    : {metrics["numerical_acc"]:>6.2f}%')
print(f'  Conceptual Q Acc   : {metrics["conceptual_acc"]:>6.2f}%')
print(f'  Macro F1 Score     : {metrics["macro_f1"]:>6.4f}')
print('=========================================================')


Evaluating: 100%|██████████| 4200/4200 [01:49<00:00, 38.37it/s]



========== EVALUATION RESULTS (ResNet50 CNN) ==========
  Overall Accuracy   :  54.62%
  Numerical Q Acc    :  17.25%
  Conceptual Q Acc   :  90.73%
  Macro F1 Score     : 0.0120


In [ ]:
evaluator.plot_confusion_matrix(metrics)
evaluator.print_classification_report(metrics)



CLASS                          PREC    REC     F1    SUP
--------------------------------------------------------------
0                             0.000  0.000  0.000     29
0 degrees                     0.000  0.000  0.000      3
1                             0.173  0.500  0.257     44
10                            0.739  0.919  0.819     37
10 degrees                    0.000  0.000  0.000     11
100 degrees                   0.000  0.000  0.000      3
10k feet                      0.000  0.000  0.000     27
110 degrees                   0.000  0.000  0.000      7
11k feet                      0.000  0.000  0.000     13
120 degrees                   0.000  0.000  0.000      8
12k feet                      0.000  0.000  0.000     15
130 degrees                   0.125  0.200  0.154     10
13k feet                      0.000  0.000  0.000      8
140 degrees                   0.000  0.000  0.000     11
14k feet                      0.000  0.000  0.000     13
150 degrees             

## 💾 Step 14 — Output Summary

In [ ]:
import os
print(f'All outputs in Google Drive:\n  {BASE_DIR}\n')
for root, dirs, files in os.walk(BASE_DIR):
    dirs[:] = [d for d in dirs if '__pycache__' not in d]
    level   = root.replace(BASE_DIR,'').count(os.sep)
    indent  = '  '*level
    folder  = os.path.basename(root)
    if files:
        sz = sum(os.path.getsize(os.path.join(root,f)) for f in files)/1024
        print(f'{indent}{folder}/  ({len(files)} files, {sz:.0f} KB)')


All outputs in Google Drive:
  /content/drive/MyDrive/aviation_vqa_output

    raw/  (2002 files, 8760 KB)
      images/  (2000 files, 29987 KB)
    processed/  (5 files, 8260 KB)
      images/  (2000 files, 29987 KB)
    annotations/  (1 files, 3356 KB)
  embeddings/  (4 files, 88330 KB)
  chromadb_store/  (1 files, 62464 KB)
    947c4ba3-c0d5-44ad-a126-5438a0a14b16/  (5 files, 29315 KB)
    b25508c4-0586-4a34-ada0-730ba49e7845/  (5 files, 29393 KB)
    63324f9f-d059-475d-85c1-ed26f83a9c9c/  (5 files, 29289 KB)
  models/  (6 files, 928137 KB)
    yolov5_aircraft/  (21 files, 3413 KB)
      weights/  (2 files, 7265 KB)
  evaluation_results/  (2 files, 5441 KB)
  yolo_aircraft/  (1 files, 0 KB)
      train/  (1700 files, 25475 KB)
      val/  (300 files, 4511 KB)
    labels/  (2 files, 593 KB)
      train/  (1700 files, 434 KB)
      val/  (300 files, 77 KB)


## 🎯 Step 16 — YOLOv5 Aircraft Detection (auto-labeled + fine-tuned)
> Your radar frames are synthetic blips, not photos, so a stock COCO YOLOv5 wouldn't recognize them. This cell auto-generates bounding-box labels from the exact blip color the renderer uses, then fine-tunes a real YOLOv5n on your radar images.
> Run after Step 4 (needs `rendered`). One-time cost: ~10-15 min to train; reruns skip training if weights already exist on Drive.

In [ ]:
# ============================================================
# STEP 16 — YOLOv5 Aircraft Blip Detection (auto-labeled + fine-tuned)
# Trains a real YOLOv5 model to detect aircraft blips on the radar
# images, since your radar frames are synthetic (not photos), so a
# stock COCO-pretrained YOLOv5 wouldn't recognize them.
#
# Labels are generated automatically from the known blip color
# (#00ff44) used by radar_generation/radar_renderer.py — no manual
# annotation needed.
#
# Run after Step 4 (needs `rendered`) and Step 2 (needs BASE_DIR).
# ============================================================

import os, sys, subprocess, random
import numpy as np
import cv2
from PIL import Image

assert "rendered" in dir(), "Run Step 4 first so `rendered` (list of frames) exists."
assert "BASE_DIR" in dir(), "Run Step 2 first so `BASE_DIR` exists."

YOLO_ROOT    = "/content/yolov5"
YOLO_DATA    = os.path.join(BASE_DIR, "yolo_aircraft")
YOLO_RUNDIR  = os.path.join(BASE_DIR, "models", "yolov5_aircraft")
YOLO_WEIGHTS = os.path.join(YOLO_RUNDIR, "weights", "best.pt")

os.makedirs(YOLO_DATA, exist_ok=True)
for split in ["images/train", "images/val", "labels/train", "labels/val"]:
    os.makedirs(os.path.join(YOLO_DATA, split), exist_ok=True)

# ------------------------------------------------------------
# 1. Clone YOLOv5 + install requirements (once)
# ------------------------------------------------------------
if not os.path.exists(YOLO_ROOT):
    print("Cloning YOLOv5 ...")
    subprocess.run(["git", "clone", "-q", "https://github.com/ultralytics/yolov5", YOLO_ROOT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                     os.path.join(YOLO_ROOT, "requirements.txt")], check=True)
    print("YOLOv5 ready  ✓")
else:
    print("YOLOv5 already cloned  ✓")

# ------------------------------------------------------------
# 2. Auto-generate YOLO labels from the exact blip color used
#    at render time (matches BLIP = "#00ff44" in radar_renderer.py)
# ------------------------------------------------------------
BLIP_RGB  = np.array([0, 255, 68])
COLOR_TOL = 40
BOX_PX    = 20  # fixed box size (px, at 224x224) drawn around each blip centroid


def find_blip_boxes(image_path, img_size=224):
    img = np.array(Image.open(image_path).convert("RGB"))
    dist = np.linalg.norm(img.astype(int) - BLIP_RGB, axis=-1)
    mask = (dist < COLOR_TOL).astype(np.uint8) * 255
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for c in contours:
        m = cv2.moments(c)
        if m["m00"] == 0:
            continue
        cx, cy = m["m10"] / m["m00"], m["m01"] / m["m00"]
        x1, y1 = max(0, cx - BOX_PX / 2), max(0, cy - BOX_PX / 2)
        x2, y2 = min(img_size, cx + BOX_PX / 2), min(img_size, cy + BOX_PX / 2)
        boxes.append((x1, y1, x2, y2))
    return boxes


def write_yolo_label(boxes, out_path, img_size=224):
    lines = []
    for x1, y1, x2, y2 in boxes:
        xc, yc = (x1 + x2) / 2 / img_size, (y1 + y2) / 2 / img_size
        w, h = (x2 - x1) / img_size, (y2 - y1) / img_size
        lines.append(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
    with open(out_path, "w") as f:
        f.write("\n".join(lines))


if not os.listdir(os.path.join(YOLO_DATA, "images/train")):
    print("Auto-labeling radar images from blip color ...")
    frames = [f for f in rendered if os.path.exists(f["image_path"])]
    random.seed(42)
    random.shuffle(frames)
    cut = int(len(frames) * 0.85)
    splits = {"train": frames[:cut], "val": frames[cut:]}

    for split, split_frames in splits.items():
        for fr in split_frames:
            stem = fr["frame_id"]
            img_dst = os.path.join(YOLO_DATA, f"images/{split}/{stem}.png")
            lbl_dst = os.path.join(YOLO_DATA, f"labels/{split}/{stem}.txt")
            if not os.path.exists(img_dst):
                Image.open(fr["image_path"]).save(img_dst)
            boxes = find_blip_boxes(fr["image_path"])
            write_yolo_label(boxes, lbl_dst)
    print(f"Labeled {len(frames)} images  ✓  (train={len(splits['train'])}, val={len(splits['val'])})")
else:
    print("Labeled dataset already exists on Drive  ✓")

data_yaml = os.path.join(YOLO_DATA, "data.yaml")
with open(data_yaml, "w") as f:
    f.write(
        f"train: {os.path.join(YOLO_DATA, 'images/train')}\n"
        f"val: {os.path.join(YOLO_DATA, 'images/val')}\n"
        f"nc: 1\n"
        f"names: ['aircraft']\n"
    )

# ------------------------------------------------------------
# 3. Fine-tune YOLOv5n on the labeled radar images
#    (skips automatically if weights already exist on Drive)
# ------------------------------------------------------------
if os.path.exists(YOLO_WEIGHTS):
    print(f"Trained weights already found on Drive  ✓  {YOLO_WEIGHTS}")
else:
    print("Training YOLOv5n on radar blips (roughly 10-15 min on a Colab GPU)...")
    subprocess.run([
        sys.executable, os.path.join(YOLO_ROOT, "train.py"),
        "--img", "224", "--batch", "32", "--epochs", "25",
        "--data", data_yaml,
        "--weights", "yolov5n.pt",
        "--project", os.path.dirname(YOLO_RUNDIR),
        "--name", os.path.basename(YOLO_RUNDIR),
        "--exist-ok",
    ], cwd=YOLO_ROOT, check=True)
    print(f"Training complete  ✓  weights saved to {YOLO_WEIGHTS}")


Cloning YOLOv5 ...
YOLOv5 ready  ✓
Labeled dataset already exists on Drive  ✓
Trained weights already found on Drive  ✓  /content/drive/MyDrive/aviation_vqa_output/models/yolov5_aircraft/weights/best.pt


## 🖼️ Step 17 — YOLOv5 Detection Panel
> Draws a box around every aircraft blip YOLOv5 has identified, with a confidence score. Run after Step 16.

In [ ]:
# ============================================================
# STEP 17 — YOLOv5 Detection Panel
# Draws a box around every aircraft blip YOLOv5 has identified
# on a radar image, with a confidence score per detection.
# Run after Step 16.
# ============================================================

import os, sys, random
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image, ImageDraw

assert "YOLO_WEIGHTS" in dir(), "Run Step 16 first so the model is trained."
assert os.path.exists(YOLO_WEIGHTS), "YOLOv5 weights not found — run Step 16 first."

# --- fix a namespace collision: your aviation_vqa project has its own
# `models` package (models/vqa_model.py etc.), which gets found instead of
# YOLOv5's own `models/common.py` unless we clear the cached module first
# and make sure YOLOv5's repo is searched before aviation_vqa's.
for _mod in list(sys.modules):
    if _mod == "models" or _mod.startswith("models."):
        del sys.modules[_mod]
if YOLO_ROOT in sys.path:
    sys.path.remove(YOLO_ROOT)
sys.path.insert(0, YOLO_ROOT)

yolo_model = torch.hub.load(YOLO_ROOT, "custom", path=YOLO_WEIGHTS, source="local")
yolo_model.conf = 0.25
print("YOLOv5 aircraft detector loaded  ✓")

img_files = [f["image_path"] for f in rendered if os.path.exists(f["image_path"])]

detect_out = widgets.Output()
new_btn    = widgets.Button(description="🔀 New Image", layout=widgets.Layout(width="150px"))
detect_btn = widgets.Button(description="🎯 Detect Aircraft", button_style="success",
                             layout=widgets.Layout(width="170px"))
conf_slider = widgets.FloatSlider(value=0.25, min=0.05, max=0.9, step=0.05,
                                   description="Min conf:")

state17 = {"img_path": random.choice(img_files) if img_files else None}


def draw_detections(img_path, upscale=2):
    yolo_model.conf = conf_slider.value
    results = yolo_model(img_path, size=224)
    df = results.pandas().xyxy[0]
    img = Image.open(img_path).convert("RGB")
    disp = img.resize((224 * upscale, 224 * upscale))
    draw = ImageDraw.Draw(disp)
    for _, row in df.iterrows():
        x1, y1, x2, y2 = [v * upscale for v in (row.xmin, row.ymin, row.xmax, row.ymax)]
        draw.rectangle([x1, y1, x2, y2], outline="#ff3333", width=2)
        draw.text((x1, max(0, y1 - 11)), f"aircraft {row.confidence:.2f}", fill="#ff3333")
    return disp, len(df)


def render17():
    with detect_out:
        clear_output(wait=True)
        if not state17["img_path"]:
            print("No rendered images found.")
            return
        disp, n = draw_detections(state17["img_path"])
        display(disp)
        print(f"{os.path.basename(state17['img_path'])}  —  {n} aircraft detected")


def on_new17(_):
    state17["img_path"] = random.choice(img_files)
    render17()


def on_detect17(_):
    render17()


new_btn.on_click(on_new17)
detect_btn.on_click(on_detect17)
conf_slider.observe(lambda _: render17(), names="value")

display(widgets.VBox([
    widgets.HTML("<h3>🎯 YOLOv5 Aircraft Detection Panel</h3>"),
    widgets.HBox([new_btn, detect_btn, conf_slider]),
    detect_out,
]))
render17()


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
YOLOv5 🚀 v7.0-550-g66fa8458 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 68 layers, 1,760,518 parameters, 0 gradients, 4.1 GFLOPs
Adding AutoShape... 
YOLOv5 aircraft detector loaded  ✓


## 🤖 Step 18 — Relevance Classifier (open-source LLM, free)
> Uses **Qwen2.5-1.5B-Instruct** (Apache 2.0, runs locally in Colab — no API key, no cost, nothing leaves the notebook) to decide whether a question is actually answerable from the radar image before it reaches the CNN model or the flight locator. Falls back to a keyword heuristic if the model's output can't be parsed cleanly.


In [ ]:
# ============================================================
# STEP 18 — Relevance Classifier (open-source LLM, free, local)
# Uses Qwen2.5-1.5B-Instruct (Apache 2.0, runs free on a Colab
# GPU or CPU) to decide whether a question is actually something
# this radar VQA system can answer, before it reaches cnn_engine
# or the flight locator. No API key, no cost, nothing leaves Colab.
# ============================================================

import subprocess, sys

try:
    import transformers  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                            "transformers", "accelerate"])

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

_RELEVANCE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # small, free, Apache-2.0, good instruction following
_device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {_RELEVANCE_MODEL_ID} ({_device}) — first run downloads ~3GB, then it's cached...")
_relevance_tokenizer = AutoTokenizer.from_pretrained(_RELEVANCE_MODEL_ID)
_relevance_model = AutoModelForCausalLM.from_pretrained(
    _RELEVANCE_MODEL_ID,
    torch_dtype=torch.float16 if _device == "cuda" else torch.float32,
).to(_device)
_relevance_model.eval()
print("Relevance LLM ready  ✓")

_RELEVANCE_SYSTEM_PROMPT = """You are a strict relevance filter in front of an aviation radar \
Visual Question Answering system. The system only sees a single 224x224 synthetic PPI radar image \
containing a handful of aircraft "blips" — it can answer questions about: how many aircraft are \
visible, an aircraft's position (left/right/top/bottom/quadrant/center), altitude, heading, callsign, \
comparisons between aircraft (highest/lowest altitude etc.), and locating/highlighting a specific \
aircraft by callsign.

It CANNOT answer anything requiring outside knowledge, real-world facts, opinions, math unrelated to \
the image, or general conversation.

Reply with exactly one word: RELEVANT or IRRELEVANT. No punctuation, no explanation."""


def _keyword_fallback(question: str) -> bool:
    """Used only if the local model's output can't be parsed cleanly."""
    kw = ["aircraft", "flight", "blip", "radar", "altitude", "heading", "callsign",
          "left", "right", "top", "bottom", "center", "quadrant", "how many",
          "locate", "find", "highlight", "position", "coordinate"]
    q = question.lower()
    return any(k in q for k in kw)


@torch.no_grad()
def classify_relevance(question: str) -> bool:
    """Returns True if the question is something the radar VQA system should try to answer."""
    messages = [
        {"role": "system", "content": _RELEVANCE_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    prompt = _relevance_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = _relevance_tokenizer(prompt, return_tensors="pt").to(_device)

    try:
        out = _relevance_model.generate(
            **inputs, max_new_tokens=4, do_sample=False,
            pad_token_id=_relevance_tokenizer.eos_token_id,
        )
        new_tokens = out[0][inputs["input_ids"].shape[1]:]
        verdict = _relevance_tokenizer.decode(new_tokens, skip_special_tokens=True).strip().upper()
        if "IRRELEVANT" in verdict:
            return False
        if "RELEVANT" in verdict:
            return True
        # Ambiguous output — fall back to keyword heuristic
        return _keyword_fallback(question)
    except Exception as e:
        print(f"[relevance] local LLM inference failed ({e}); using keyword fallback.")
        return _keyword_fallback(question)


print("classify_relevance() ready  ✓  (Qwen2.5-1.5B-Instruct, local, free)")


Loading Qwen/Qwen2.5-1.5B-Instruct (cuda) — first run downloads ~3GB, then it's cached...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Relevance LLM ready  ✓
classify_relevance() ready  ✓  (Qwen2.5-1.5B-Instruct, local, free)


## 📍 Step 19 — Flight Locator (coordinates + YOLO highlight)
> Given a callsign, retrieves its real recorded coordinates (latitude/longitude/altitude/heading) and uses the fine-tuned YOLOv5 detector (Step 16) to highlight that specific aircraft's blip. Run after Step 4 and Step 16.


In [ ]:
# ============================================================
# STEP 19 — Flight Locator (coordinates + YOLO highlight)
# Given a callsign, retrieves its real recorded coordinates and
# uses the fine-tuned YOLOv5 detector to highlight that specific
# aircraft's blip on its radar image.
#
# Run after Step 4 (needs `rendered`) and Step 16 (needs
# YOLO_ROOT / YOLO_WEIGHTS; loads its own yolo_model if Step 17
# hasn't been run yet).
# ============================================================

import os, re, sys, difflib
import torch
from PIL import Image, ImageDraw

assert "rendered" in dir(), "Run Step 4 first so `rendered` exists."
assert "YOLO_WEIGHTS" in dir() and os.path.exists(YOLO_WEIGHTS), "Run Step 16 first to train the detector."

sys.path.insert(0, '/content/aviation_vqa')
from radar_generation.radar_renderer import RadarRenderer

_renderer_for_coords = RadarRenderer()  # only used for its get_norm_coords() math, no rendering

if "yolo_model" not in dir():
    # --- same `models` package collision fix as Step 17 (see there for why)
    for _mod in list(sys.modules):
        if _mod == "models" or _mod.startswith("models."):
            del sys.modules[_mod]
    if YOLO_ROOT in sys.path:
        sys.path.remove(YOLO_ROOT)
    sys.path.insert(0, YOLO_ROOT)

    yolo_model = torch.hub.load(YOLO_ROOT, "custom", path=YOLO_WEIGHTS, source="local")
    yolo_model.conf = 0.25
    print("YOLOv5 detector loaded for flight locator  ✓")

# ------------------------------------------------------------
# 1. Build a callsign -> record index, with exact render
#    coordinates (nx, ny) recomputed via the same normalisation
#    math the renderer used when it originally drew the image.
# ------------------------------------------------------------
callsign_index = {}
for frame in rendered:
    if not os.path.exists(frame["image_path"]):
        continue
    enriched = _renderer_for_coords.get_norm_coords(frame["aircraft"])
    for ac in enriched:
        cs = ac["callsign"].strip().upper()
        callsign_index.setdefault(cs, []).append({
            "frame_id":  frame["frame_id"],
            "image_path": frame["image_path"],
            "callsign":  ac["callsign"],
            "icao24":    ac.get("icao24"),
            "latitude":  ac["latitude"],
            "longitude": ac["longitude"],
            "altitude":  ac.get("altitude"),
            "heading":   ac.get("heading"),
            "nx": ac["nx"], "ny": ac["ny"],
        })

print(f"Callsign index built  ✓  ({len(callsign_index)} unique callsigns across {len(rendered)} frames)")

_CALLSIGN_PATTERN = re.compile(r"\b[A-Za-z]{2,5}\d{1,5}[A-Za-z]?\b")


def extract_callsign(question: str):
    """Pull the most plausible callsign token out of a free-text question."""
    candidates = _CALLSIGN_PATTERN.findall(question)
    if not candidates:
        return None
    for c in candidates:
        if c.upper() in callsign_index:
            return c.upper()
    for c in candidates:
        match = difflib.get_close_matches(c.upper(), callsign_index.keys(), n=1, cutoff=0.6)
        if match:
            return match[0]
    return None


def locate_flight(callsign: str):
    """Returns the first known record for a callsign, or None if not found."""
    matches = callsign_index.get(callsign.upper())
    return matches[0] if matches else None


# ------------------------------------------------------------
# 2. Highlight that aircraft's blip using the trained YOLOv5
#    detector — match the detection nearest the aircraft's known
#    render position, rather than trusting pixel math alone,
#    since matplotlib's tight-bbox crop shifts coordinates
#    slightly frame to frame.
# ------------------------------------------------------------
def highlight_flight(record: dict, upscale: int = 2, img_size: int = 224):
    img = Image.open(record["image_path"]).convert("RGB")
    disp = img.resize((img_size * upscale, img_size * upscale))
    draw = ImageDraw.Draw(disp)

    results = yolo_model(record["image_path"], size=img_size)
    df = results.pandas().xyxy[0]

    # Expected pixel location from the known (nx, ny); note the image's
    # row 0 is the TOP of the figure, so the y-axis must be flipped.
    target_x = record["nx"] * img_size
    target_y = (1 - record["ny"]) * img_size

    matched = None
    best_dist = float("inf")
    for _, row in df.iterrows():
        cx, cy = (row.xmin + row.xmax) / 2, (row.ymin + row.ymax) / 2
        d = (cx - target_x) ** 2 + (cy - target_y) ** 2
        if d < best_dist:
            best_dist, matched = d, row

    for _, row in df.iterrows():
        x1, y1, x2, y2 = [v * upscale for v in (row.xmin, row.ymin, row.xmax, row.ymax)]
        is_match = matched is not None and row.equals(matched)
        color = "#ffcc00" if is_match else "#888888"
        width = 4 if is_match else 1
        draw.rectangle([x1, y1, x2, y2], outline=color, width=width)
        if is_match:
            draw.text((x1, max(0, y1 - 14)), f"🎯 {record['callsign']}", fill="#ffcc00")

    found = matched is not None
    return disp, found


def locate_and_highlight(question: str):
    """
    End-to-end: parse a callsign out of the question, look up its real
    coordinates, and return an annotated image + a text answer.
    Returns None if no callsign could be resolved.
    """
    callsign = extract_callsign(question)
    if not callsign:
        return None
    record = locate_flight(callsign)
    if not record:
        return {
            "found": False,
            "answer": f"I don't have a record of a flight called {callsign}.",
            "image": None,
        }
    disp, matched_in_image = highlight_flight(record)
    coord_text = (
        f"{record['callsign']} is at latitude {record['latitude']:.4f}, "
        f"longitude {record['longitude']:.4f}, altitude {record['altitude']} ft, "
        f"heading {record['heading']}°."
    )
    if not matched_in_image:
        coord_text += " (YOLO didn't detect a matching blip on this image, showing its known position only.)"
    return {"found": True, "answer": coord_text, "image": disp, "record": record}


print("Flight locator ready  ✓  — try locate_and_highlight('locate DAL1234')")


Callsign index built  ✓  (14031 unique callsigns across 2000 frames)
Flight locator ready  ✓  — try locate_and_highlight('locate DAL1234')


## 🧪 Step 20a — Answer Evaluation & Fallback
> Wraps `cnn_engine.answer()` with a confidence threshold, a YOLO-count cross-check for counting questions, and a hedge/decline fallback — so low-trust answers are never stated as flat fact. Every flagged case is logged to Drive: one log for future retraining, one escalation log that stands in for what a real staff-notification system (email/Slack/ticketing) would read from. Run after `cnn_engine` and `yolo_model` both exist.


In [ ]:
# ============================================================
# STEP 20a — Answer Evaluation & Fallback
# Wraps cnn_engine.answer() with:
#   1. Confidence thresholding      -> hedge or decline low-confidence answers
#   2. YOLO count cross-check       -> catches counting answers YOLO itself disagrees with
#   3. Hedge / decline fallback     -> never state a low-trust answer as fact
#   4. Logging                      -> every flagged case written to Drive for
#                                       (a) future retraining and (b) an
#                                       escalation queue a real staff-notification
#                                       system (email/Slack/ticketing) could read from
#
# Run after cnn_engine and yolo_model both exist (Steps 16/17 or 19, and the
# CNN pipeline).
# ============================================================

import os, re, json, datetime
from PIL import Image

assert "cnn_engine" in dir(), "Run the CNN inference cell first."
assert "yolo_model" in dir(), "Run Step 17 or Step 19 first so yolo_model exists."

CONF_HEDGE_THRESHOLD   = 0.55  # below this -> hedge ("I think... but not fully sure")
CONF_DECLINE_THRESHOLD = 0.30  # below this -> decline outright, don't guess
YOLO_COUNT_TOLERANCE   = 1     # allowed difference between CNN count and YOLO's own count

LOG_DIR = os.path.join(BASE_DIR, "logs")
os.makedirs(LOG_DIR, exist_ok=True)
RETRAIN_LOG    = os.path.join(LOG_DIR, "low_confidence_for_retraining.jsonl")
ESCALATION_LOG = os.path.join(LOG_DIR, "escalations.jsonl")  # stand-in for a real
                                                               # staff-notification queue

_COUNT_QUESTION = re.compile(r"\bhow many\b|\bnumber of\b|\bcount\b", re.IGNORECASE)


def _extract_int(text):
    m = re.search(r"\d+", str(text))
    return int(m.group()) if m else None


def _yolo_count(image_path, conf=0.25):
    """Independent aircraft count from YOLO, used to sanity-check counting answers."""
    prev_conf = yolo_model.conf
    yolo_model.conf = conf
    try:
        results = yolo_model(image_path, size=224)
        return len(results.pandas().xyxy[0])
    finally:
        yolo_model.conf = prev_conf


def _log_jsonl(path, record):
    record = {**record, "timestamp": datetime.datetime.utcnow().isoformat()}
    with open(path, "a") as f:
        f.write(json.dumps(record) + "\n")


def evaluate_and_answer(image_path: str, question: str) -> dict:
    """
    Runs the CNN VQA model, cross-checks numerical questions against YOLO's
    own detection count, and decides whether to answer normally, hedge, or
    decline — logging anything uncertain for retraining and, for outright
    declines, for staff escalation.

    Returns: {"answer": str, "raw_answer": str, "confidence": float,
              "verdict": str, "note": str or None}
    """
    img = Image.open(image_path).convert("RGB")
    result = cnn_engine.answer(img, question)
    answer, confidence = result["answer"], result.get("confidence", 0.0)

    verdict, note = "confident", None

    # --- 1. YOLO cross-check, only for counting-style questions ---
    if _COUNT_QUESTION.search(question):
        predicted_n = _extract_int(answer)
        if predicted_n is not None:
            yolo_n = _yolo_count(image_path)
            if abs(predicted_n - yolo_n) > YOLO_COUNT_TOLERANCE:
                verdict = "yolo_mismatch"
                note = f"YOLO detected {yolo_n} aircraft, model predicted {predicted_n}"

    # --- 2. confidence thresholds (only checked if YOLO didn't already flag it) ---
    if verdict == "confident":
        if confidence < CONF_DECLINE_THRESHOLD:
            verdict = "low_confidence_decline"
        elif confidence < CONF_HEDGE_THRESHOLD:
            verdict = "low_confidence_hedge"

    # --- 3. decide the final answer text ---
    if verdict == "low_confidence_decline":
        final_answer = ("I'm not confident enough in an answer to that one — "
                         "it's been flagged for review rather than guessed.")
        _log_jsonl(ESCALATION_LOG, {
            "question": question, "image_path": image_path,
            "model_answer": answer, "confidence": confidence,
            "reason": verdict, "note": note,
        })
    elif verdict == "yolo_mismatch":
        final_answer = f"{answer} — but I'm not fully sure ({note}); flagged for review."
        _log_jsonl(ESCALATION_LOG, {
            "question": question, "image_path": image_path,
            "model_answer": answer, "confidence": confidence,
            "reason": verdict, "note": note,
        })
    elif verdict == "low_confidence_hedge":
        final_answer = f"{answer} — I think, though I'm only moderately confident."
    else:
        final_answer = answer

    # --- 4. log every non-confident case for future retraining ---
    if verdict != "confident":
        _log_jsonl(RETRAIN_LOG, {
            "question": question, "image_path": image_path,
            "model_answer": answer, "confidence": confidence,
            "reason": verdict, "note": note,
        })

    return {"answer": final_answer, "raw_answer": answer,
            "confidence": confidence, "verdict": verdict, "note": note}


print("evaluate_and_answer() ready  ✓  (confidence threshold + YOLO cross-check + hedge/decline + logging)")
print(f"  Retraining log:  {RETRAIN_LOG}")
print(f"  Escalation log:  {ESCALATION_LOG}")


evaluate_and_answer() ready  ✓  (confidence threshold + YOLO cross-check + hedge/decline + logging)
  Retraining log:  /content/drive/MyDrive/aviation_vqa_output/logs/low_confidence_for_retraining.jsonl
  Escalation log:  /content/drive/MyDrive/aviation_vqa_output/logs/escalations.jsonl


## 🧭 Step 20 — Unified Query Handler
> Routes every question through: LLM relevance check (18) → locate/highlight (19) → evaluated CNN VQA (20a: confidence threshold + YOLO cross-check + hedge/decline + logging). This is what both the voice panel and any future text panel should call.


In [ ]:
# ============================================================
# STEP 20 — Unified Query Handler
# Every question (voice or text) now flows through one place:
#   1. LLM relevance check (Step 18)          -> reject if irrelevant
#   2. Locate/highlight check (Step 19)       -> if it's a "locate
#      flight <callsign>" question, run YOLO + coordinate lookup
#   3. Otherwise                              -> evaluate_and_answer() (Step 20a):
#      cnn_engine (ResNet50) + confidence threshold + YOLO count
#      cross-check + hedge/decline fallback + logging
# Run after Steps 18, 19, 20a, and the CNN pipeline (`cnn_engine`).
# ============================================================

import re

assert "classify_relevance" in dir(), "Run Step 18 first."
assert "locate_and_highlight" in dir(), "Run Step 19 first."
assert "evaluate_and_answer" in dir(), "Run Step 20a first."

_LOCATE_TRIGGER = re.compile(r"\b(locate|find|where\s+is|highlight|show\s+me)\b", re.IGNORECASE)


def handle_question(image_path: str, question: str) -> dict:
    """
    Returns a dict always shaped like:
    {
        "kind": "irrelevant" | "locate" | "vqa",
        "answer": str,
        "image": PIL.Image or None,   # only set for "locate" (annotated image)
        "confidence": float or None,
        "verdict": str or None,       # only set for "vqa": confident /
                                       # low_confidence_hedge / low_confidence_decline /
                                       # yolo_mismatch
    }
    """
    # 1. Relevance filter
    if not classify_relevance(question):
        return {
            "kind": "irrelevant",
            "answer": "That's not something I can answer from this radar image — "
                      "try asking about aircraft count, position, altitude, heading, or a callsign.",
            "image": None,
            "confidence": None,
            "verdict": None,
        }

    # 2. Locate / highlight a specific flight
    if _LOCATE_TRIGGER.search(question):
        result = locate_and_highlight(question)
        if result is not None:
            return {
                "kind": "locate",
                "answer": result["answer"],
                "image": result["image"],
                "confidence": None,
                "verdict": None,
            }
        # No callsign could be parsed out of a "locate"-style question —
        # fall through to the normal VQA path below.

    # 3. Normal VQA, with confidence thresholding + YOLO cross-check + fallback
    result = evaluate_and_answer(image_path, question)
    return {
        "kind": "vqa",
        "answer": result["answer"],
        "image": None,
        "confidence": result["confidence"],
        "verdict": result["verdict"],
    }


print("Unified query handler ready  ✓  (relevance -> locate -> evaluated CNN VQA)")


Unified query handler ready  ✓  (relevance -> locate -> evaluated CNN VQA)


## 🎙️🔊 Step 21 — Voice Input + Voice Output Panel (Whisper + Speech, relevance + locate aware)
> Speak a question — including "locate flight `<callsign>`" to highlight it. Every question is filtered for relevance (Step 18), checked for a locate/highlight intent (Step 19), and otherwise answered by the ResNet50 CNN engine (no CLIP anywhere). The answer is shown in a panel **and** spoken back out loud. Run after Steps 18–20.


In [ ]:
# ============================================================
# STEP 15 — Voice Input + Voice Output Panel (Whisper + TTS)
# Every question goes through the unified handler (Step 20):
#   LLM relevance check -> locate/highlight via YOLO -> CNN VQA
# No CLIP, no ChromaDB, no hybrid routing anywhere in this cell.
#
# Speak your question -> Whisper transcribes it -> handle_question()
# resolves it -> the answer (and highlighted image, if it was a
# "locate" question) is shown in a styled panel AND spoken back out
# loud in the browser.
#
# Run AFTER Steps 18, 19, 20 (needs `handle_question`) and Step 1
# (needs faster-whisper installed).
# ============================================================

import os, sys, json, base64, subprocess, random, html
import ipywidgets as widgets
from IPython.display import display, Javascript, clear_output
from google.colab.output import eval_js
from PIL import Image

assert "handle_question" in dir(), "Run Steps 18, 19, and 20 first so `handle_question` exists."
assert "BASE_DIR" in dir(), "Run Step 2 first so `BASE_DIR` exists."

PROC_DIR = os.path.join(BASE_DIR, "data/processed")
IMG_DIR  = os.path.join(PROC_DIR, "images")

# ------------------------------------------------------------
# 1. Load the Whisper model once (faster-whisper)
# ------------------------------------------------------------
from faster_whisper import WhisperModel
import torch

_WHISPER_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_WHISPER_COMPUTE = "float16" if _WHISPER_DEVICE == "cuda" else "int8"

print("Loading Whisper model (base) ...")
_whisper_model = WhisperModel("base", device=_WHISPER_DEVICE, compute_type=_WHISPER_COMPUTE)
print(f"Whisper ready  ✓  (device={_WHISPER_DEVICE})")


def transcribe_wav(wav_path: str) -> str:
    segments, _info = _whisper_model.transcribe(wav_path, language="en", vad_filter=True)
    return " ".join(seg.text.strip() for seg in segments).strip()


# ------------------------------------------------------------
# 2. Browser microphone recorder (records webm, returns data URL)
# ------------------------------------------------------------
def record_browser_audio(seconds=5):
    display(Javascript("""
    async function recordAviationAudioPanel(seconds) {
        const stream = await navigator.mediaDevices.getUserMedia({audio: true});
        const recorder = new MediaRecorder(stream);
        const chunks = [];
        recorder.ondataavailable = e => { if (e.data.size > 0) chunks.push(e.data); };
        recorder.start();
        await new Promise(resolve => setTimeout(resolve, seconds * 1000));
        recorder.stop();
        await new Promise(resolve => recorder.onstop = resolve);
        stream.getTracks().forEach(track => track.stop());
        const blob = new Blob(chunks, {type: "audio/webm"});
        const reader = new FileReader();
        return await new Promise(resolve => {
            reader.onloadend = () => resolve(reader.result);
            reader.readAsDataURL(blob);
        });
    }
    """))
    return eval_js(f"recordAviationAudioPanel({int(seconds)})")


def save_and_convert_to_wav(data_url: str, out_wav: str):
    header, b64data = data_url.split(",", 1)
    raw = base64.b64decode(b64data)
    webm_path = out_wav.replace(".wav", ".webm")
    with open(webm_path, "wb") as f:
        f.write(raw)
    subprocess.run(
        ["ffmpeg", "-y", "-i", webm_path, "-ar", "16000", "-ac", "1", out_wav],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True,
    )
    return out_wav


# ------------------------------------------------------------
# 3. Speak text out loud using the browser's built-in TTS
# ------------------------------------------------------------
def speak_text(text: str):
    safe_text = json.dumps(text)
    display(Javascript(f"""
    (function() {{
        const msg = new SpeechSynthesisUtterance({safe_text});
        msg.rate = 1.0;
        window.speechSynthesis.cancel();
        window.speechSynthesis.speak(msg);
    }})();
    """))


# ------------------------------------------------------------
# 4. The panel UI
# ------------------------------------------------------------
img_files = [f for f in os.listdir(IMG_DIR) if f.lower().endswith(".png")]

record_btn   = widgets.Button(description="🎤 Record Question (5s)", button_style="primary",
                               layout=widgets.Layout(width="230px"))
new_img_btn  = widgets.Button(description="🔀 New Random Image", layout=widgets.Layout(width="180px"))
seconds_box  = widgets.IntSlider(value=5, min=2, max=10, step=1, description="Rec. secs:")
status_label = widgets.HTML(value="")
image_out    = widgets.Output()
panel_out    = widgets.Output()

state = {"img_path": os.path.join(IMG_DIR, random.choice(img_files)) if img_files else None}


def render_image(override_img=None):
    with image_out:
        clear_output(wait=True)
        if override_img is not None:
            display(override_img)
            print("🎯 highlighted result")
        elif state["img_path"]:
            display(Image.open(state["img_path"]).resize((260, 260)))
            print(os.path.basename(state["img_path"]))


KIND_LABEL = {
    "irrelevant": ("⚠️ NOT RELEVANT", "#b71c1c"),
    "locate":     ("🎯 FLIGHT LOCATED", "#e65100"),
    "vqa":        ("✈️ CO-PILOT ANSWER (ResNet50 CNN)", "#2e7d32"),
}


def render_panel(question=None, answer=None, confidence=None, kind="vqa", error=None):
    with panel_out:
        clear_output(wait=True)
        if error:
            display(widgets.HTML(f"""
            <div style="border:1px solid #e57373;background:#fdecea;padding:14px 18px;
                        border-radius:10px;font-family:sans-serif;color:#b71c1c;">
                ⚠️ {html.escape(error)}
            </div>"""))
            return
        label, color = KIND_LABEL.get(kind, KIND_LABEL["vqa"])
        conf_html = (f"<div style='color:#555;font-size:13px;'>Confidence: {confidence:.2f}</div>"
                     if confidence is not None else "")
        display(widgets.HTML(f"""
        <div style="border:1px solid #90caf9;background:#f4f9ff;padding:16px 20px;
                    border-radius:12px;font-family:sans-serif;max-width:520px;">
            <div style="font-size:13px;color:#1565c0;font-weight:600;">🎤 YOU ASKED</div>
            <div style="font-size:16px;margin:4px 0 12px 0;">{html.escape(question or "")}</div>
            <div style="font-size:13px;color:{color};font-weight:600;">{label}</div>
            <div style="font-size:18px;font-weight:600;margin:4px 0 6px 0;">{html.escape(str(answer))}</div>
            {conf_html}
        </div>"""))


def on_new_image_clicked(_):
    if img_files:
        state["img_path"] = os.path.join(IMG_DIR, random.choice(img_files))
        render_image()
        status_label.value = ""


def on_record_clicked(_):
    record_btn.disabled = True
    status_label.value = "<b style='color:#1565c0;'>🔴 Recording... speak your question now</b>"
    try:
        data_url = record_browser_audio(seconds_box.value)
        status_label.value = "<b style='color:#f9a825;'>🧠 Transcribing with Whisper...</b>"
        wav_path = save_and_convert_to_wav(data_url, "/content/_voice_query.wav")
        question = transcribe_wav(wav_path)

        if not question:
            render_panel(error="Didn't catch that — no speech detected. Try again.")
            status_label.value = ""
            return

        status_label.value = "<b style='color:#f9a825;'>✈️ Thinking...</b>"
        result = handle_question(state["img_path"], question)

        render_panel(question=question, answer=result["answer"],
                     confidence=result["confidence"], kind=result["kind"])
        render_image(override_img=result["image"])
        speak_text(str(result["answer"]))
        status_label.value = "<b style='color:#2e7d32;'>✓ Done — spoken answer playing</b>"
    except Exception as e:
        render_panel(error=f"{type(e).__name__}: {e}")
        status_label.value = ""
    finally:
        record_btn.disabled = False


record_btn.on_click(on_record_clicked)
new_img_btn.on_click(on_new_image_clicked)

render_image()
render_panel(question="—", answer="Press the mic button and ask a question — including "
                                   "\"locate flight <callsign>\" to highlight it.")

display(widgets.VBox([
    widgets.HTML("<h3>🎙️ Voice Co-Pilot — relevance-filtered, locate-aware, ResNet50 CNN</h3>"),
    widgets.HBox([image_out, widgets.VBox([
        widgets.HBox([record_btn, new_img_btn]),
        seconds_box,
        status_label,
    ])]),
    panel_out,
]))


Loading Whisper model (base) ...


RuntimeError: CUDA failed with error device-side assert triggered